# Analisis Transisi Nilai G1 → G2 → G3: Siswa Comeback vs Tetap Gagal
**UCI Student Performance Dataset**: Matematika sebagai data utama, Portugis sebagai pembanding.

**Cara menjalankan di Google Colab**
1. Upload notebook ini ke Colab, lalu pilih **Runtime > Run all**.
2. Jika `student-mat.csv` dan `student-por.csv` belum ada di folder kerja, sel Tahap 1 akan meminta upload. Nama file harus persis sama.
3. Grafik (PNG 300 dpi), tabel (CSV), Sankey (HTML), dan kesimpulan (TXT) tersimpan di folder `hasil_analisis`, lalu diunduh sebagai zip di sel terakhir. Waktu jalan sekitar 2 sampai 4 menit.

**Kategori siswa** (populasi fokus G1 < 10, lulus jika nilai ≥ 10)

| Kategori | G1 | G2 | G3 | Keterangan |
|---|---|---|---|---|
| Comeback Awal | gagal | lulus | lulus | Bangkit sejak periode 2 |
| Comeback Akhir | gagal | gagal | lulus | Bangkit di periode akhir |
| Comeback Sementara | gagal | lulus | gagal | Sempat bangkit di periode 2, lalu gagal lagi |
| Tetap Gagal | gagal | gagal | gagal | Tidak pernah lulus |

Siswa dengan G1 ≥ 10 diberi label **Stabil Lulus** atau **Tergelincir** hanya untuk gambaran transisi keseluruhan.

**Analisis utama**: Comeback (Awal + Akhir, lulus di G3) vs Tetap Gagal. Comeback Sementara ditampilkan secara deskriptif dan ikut uji antar kategori hanya jika jumlahnya memenuhi `MIN_N_KELOMPOK`.

**Aturan anti data leakage**
- G1, G2, G3 membentuk label, jadi tidak pernah diuji sebagai faktor. G1 hanya dipakai sebagai variabel kontrol, G2 hanya sebagai checkpoint terpisah.
- Semua ambang keputusan ditetapkan di sel konfigurasi sebelum analisis dijalankan.
- Penggabungan kategori jarang hanya berdasarkan frekuensi, tanpa melihat hasil comeback.
- Model peringatan dini hanya memakai informasi yang tersedia saat G1 keluar. Seluruh praproses berada di dalam pipeline sehingga dipelajari ulang di setiap fold validasi silang.
- Semua temuan adalah **asosiasi, bukan sebab-akibat**.

## Tahap 0. Setup dan konfigurasi
Semua ambang keputusan ditulis di sini **sebelum** analisis dijalankan. Mengubahnya setelah melihat hasil termasuk data snooping.

In [ ]:
import os, shutil, warnings
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
import plotly.graph_objects as go
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, recall_score, precision_score
from sklearn.inspection import permutation_importance
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 90)

CONFIG = {
    'AMBANG_LULUS': 10,        # lulus jika nilai >= 10 (skala 0 sampai 20, sistem Portugal)
    'ALPHA': 0.05,             # batas q-value setelah koreksi Benjamini-Hochberg
    'ALPHA_INDIKASI': 0.10,    # q antara 0,05 dan 0,10 disebut "indikasi"
    'EFEK_MIN': 0.20,          # ukuran efek minimal agar dianggap bermakna
    'EFEK_REPLIKASI': 0.10,    # efek minimal di dataset lain / strata agar dianggap searah
    'STABILITAS_MIN': 0.70,    # proporsi bootstrap minimal agar faktor dianggap stabil
    'N_BOOTSTRAP': 1000,
    'N_PERMUTASI': 5000,       # untuk chi-square dengan frekuensi harapan < 5
    'MIN_N_KATEGORI': 10,      # kategori nominal lebih kecil dari ini digabung ke "other"
    'MIN_N_KELOMPOK': 10,      # kategori siswa lebih kecil dari ini tidak ikut uji antar kategori
    'BATAS_DOMINAN': 0.95,     # faktor dengan satu nilai > 95% dikeluarkan (hampir tanpa variasi)
    'CV_FOLD': 5,
    'CV_ULANG': 10,
    'C_LOGISTIK': 0.5,         # kekuatan regularisasi L2, ditetapkan di awal dan tidak dituning
    'SERTAKAN_SEMENTARA_DI_BINER': False,  # True: Comeback Sementara masuk kelompok gagal
    'SEED': 42,
}
A, AI, E = CONFIG['ALPHA'], CONFIG['ALPHA_INDIKASI'], CONFIG['EFEK_MIN']
OUT = 'hasil_analisis'
os.makedirs(OUT, exist_ok=True)
RNG = np.random.default_rng(CONFIG['SEED'])

URUTAN_FOKUS = ['Comeback Awal', 'Comeback Akhir', 'Comeback Sementara', 'Tetap Gagal']
URUTAN_SEMUA = URUTAN_FOKUS + ['Stabil Lulus', 'Tergelincir']
WARNA = {'Comeback Awal': '#2E7D32', 'Comeback Akhir': '#81C784', 'Comeback Sementara': '#F9A825',
         'Tetap Gagal': '#C62828', 'Stabil Lulus': '#1565C0', 'Tergelincir': '#8E24AA',
         'Comeback': '#2E7D32', 'Gagal': '#C62828', 'Matematika': '#1565C0', 'Portugis': '#EF6C00'}
WARNA_STATUS = {'Lolos': '#2E7D32', 'Indikasi': '#F9A825', 'Tidak bermakna': '#9E9E9E'}

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
                     'axes.titleweight': 'bold', 'axes.titlesize': 12})


def simpan(fig, nama):
    fig.savefig(os.path.join(OUT, f'{nama}.png'))
    plt.show()
    plt.close(fig)


def simpan_tabel(df, nama):
    df.to_csv(os.path.join(OUT, f'{nama}.csv'), index=False)


def angka(x, d=2):
    return '-' if pd.isna(x) else f'{x:.{d}f}'.replace('.', ',')


def pct(x, d=0):
    return '-' if pd.isna(x) else f'{100 * x:.{d}f}%'.replace('.', ',')


def fmt_q(q):
    return '-' if pd.isna(q) else ('< 0,001' if q < 0.001 else angka(q, 3))


def teks_p(nama, v):
    # menulis 'p < 0,001' atau 'p = 0,023' agar enak dibaca di kalimat
    return f'{nama} -' if pd.isna(v) else (f'{nama} < 0,001' if v < 0.001 else f'{nama} = {angka(v, 3)}')


def label_kategori(df, urut):
    n = df.kategori.value_counts()
    return [f'{k}\n(n = {n.get(k, 0)})' for k in urut]


def daftar(xs):
    xs = list(xs)
    if not xs:
        return 'tidak ada'
    return xs[0] if len(xs) == 1 else ', '.join(xs[:-1]) + ' dan ' + xs[-1]


print('Setup selesai. Output disimpan di:', os.path.abspath(OUT))

## Tahap 1. Memuat dan memahami dataset
**Tujuan:** mengenali struktur, tipe, dan peran setiap kolom. Pemisah kolom (koma atau titik koma) dideteksi otomatis karena file UCI asli memakai titik koma.

In [ ]:
FILES = {'Matematika': 'student-mat.csv', 'Portugis': 'student-por.csv'}
KODE = {'Matematika': 'mat', 'Portugis': 'por'}
LAIN = {'Matematika': 'Portugis', 'Portugis': 'Matematika'}


def pastikan_file():
    hilang = [f for f in FILES.values() if not os.path.exists(f)]
    if not hilang:
        return
    try:
        from google.colab import files
        print('Silakan upload:', ', '.join(hilang))
        files.upload()
    except ImportError:
        raise FileNotFoundError(f'File tidak ditemukan: {hilang}. Letakkan di folder yang sama dengan notebook.')
    hilang = [f for f in FILES.values() if not os.path.exists(f)]
    if hilang:
        raise FileNotFoundError(f'Nama file harus persis sama: {hilang}')


def baca_csv(path):
    with open(path, 'r', encoding='utf-8-sig') as fh:
        header = fh.readline()
    sep = ';' if header.count(';') > header.count(',') else ','
    df = pd.read_csv(path, sep=sep)
    df.columns = [str(c).strip().strip('"') for c in df.columns]
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip().str.strip('"')
    return df, sep


pastikan_file()
DATA_MENTAH = {}
for nama, f in FILES.items():
    DATA_MENTAH[nama], sep = baca_csv(f)
    print(f'{nama:<11}: {DATA_MENTAH[nama].shape[0]} baris x {DATA_MENTAH[nama].shape[1]} kolom (pemisah "{sep}")')

In [ ]:
# Kamus data: arti, tipe, dan peran setiap kolom dalam analisis
KAMUS = pd.DataFrame([
    ('school', 'Sekolah (GP / MS)', 'biner', 'latar belakang'),
    ('sex', 'Jenis kelamin (F / M)', 'biner', 'latar belakang'),
    ('age', 'Usia (15 sampai 22)', 'numerik', 'latar belakang'),
    ('address', 'Tempat tinggal (U urban / R rural)', 'biner', 'latar belakang'),
    ('famsize', 'Ukuran keluarga (LE3 <= 3 / GT3 > 3)', 'biner', 'latar belakang'),
    ('Pstatus', 'Status orang tua (T bersama / A terpisah)', 'biner', 'latar belakang'),
    ('Medu', 'Pendidikan ibu (0 sampai 4)', 'ordinal', 'latar belakang'),
    ('Fedu', 'Pendidikan ayah (0 sampai 4)', 'ordinal', 'latar belakang'),
    ('Mjob', 'Pekerjaan ibu', 'nominal', 'latar belakang'),
    ('Fjob', 'Pekerjaan ayah', 'nominal', 'latar belakang'),
    ('reason', 'Alasan memilih sekolah', 'nominal', 'latar belakang'),
    ('guardian', 'Wali siswa', 'nominal', 'latar belakang'),
    ('traveltime', 'Waktu tempuh ke sekolah (1 sampai 4)', 'ordinal', 'latar belakang'),
    ('studytime', 'Waktu belajar per minggu (1 sampai 4)', 'ordinal', 'dapat diintervensi'),
    ('failures', 'Riwayat tidak naik kelas (0 sampai 3)', 'ordinal', 'latar belakang'),
    ('schoolsup', 'Bimbingan tambahan dari sekolah', 'biner', 'dapat diintervensi'),
    ('famsup', 'Dukungan belajar dari keluarga', 'biner', 'dapat diintervensi'),
    ('paid', 'Les berbayar mata pelajaran ini', 'biner', 'dapat diintervensi'),
    ('activities', 'Ekstrakurikuler', 'biner', 'dapat diintervensi'),
    ('nursery', 'Pernah ikut TK', 'biner', 'latar belakang'),
    ('higher', 'Ingin lanjut kuliah', 'biner', 'dapat diintervensi'),
    ('internet', 'Akses internet di rumah', 'biner', 'dapat diintervensi'),
    ('romantic', 'Sedang berpacaran', 'biner', 'dapat diintervensi'),
    ('famrel', 'Kualitas hubungan keluarga (1 sampai 5)', 'ordinal', 'latar belakang'),
    ('freetime', 'Waktu luang (1 sampai 5)', 'ordinal', 'dapat diintervensi'),
    ('goout', 'Frekuensi keluar bersama teman (1 sampai 5)', 'ordinal', 'dapat diintervensi'),
    ('Dalc', 'Konsumsi alkohol hari kerja (1 sampai 5)', 'ordinal', 'dapat diintervensi'),
    ('Walc', 'Konsumsi alkohol akhir pekan (1 sampai 5)', 'ordinal', 'dapat diintervensi'),
    ('health', 'Kondisi kesehatan (1 sampai 5)', 'ordinal', 'latar belakang'),
    ('absences', 'Jumlah ketidakhadiran', 'numerik', 'dapat diintervensi'),
    ('G1', 'Nilai periode 1 (0 sampai 20)', 'nilai', 'pembentuk label dan kontrol'),
    ('G2', 'Nilai periode 2 (0 sampai 20)', 'nilai', 'pembentuk label dan checkpoint'),
    ('G3', 'Nilai akhir (0 sampai 20)', 'nilai', 'pembentuk label'),
], columns=['kolom', 'arti', 'tipe', 'peran'])

TIPE = dict(zip(KAMUS.kolom, KAMUS.tipe))
PERAN = dict(zip(KAMUS.kolom, KAMUS.peran))
FAKTOR = KAMUS.loc[KAMUS.tipe != 'nilai', 'kolom'].tolist()
LEVEL_POSITIF = {'school': 'MS', 'sex': 'M', 'address': 'U', 'famsize': 'GT3', 'Pstatus': 'T'}
LEVEL_NEGATIF = {'school': 'GP', 'sex': 'F', 'address': 'R', 'famsize': 'LE3', 'Pstatus': 'A'}
for f in FAKTOR:
    if TIPE[f] == 'biner':
        LEVEL_POSITIF.setdefault(f, 'yes')
        LEVEL_NEGATIF.setdefault(f, 'no')

NAMA = {'school': 'sekolah', 'sex': 'jenis kelamin', 'age': 'usia', 'address': 'tempat tinggal',
        'famsize': 'ukuran keluarga', 'Pstatus': 'status tinggal orang tua', 'Medu': 'pendidikan ibu',
        'Fedu': 'pendidikan ayah', 'Mjob': 'pekerjaan ibu', 'Fjob': 'pekerjaan ayah',
        'reason': 'alasan memilih sekolah', 'guardian': 'wali siswa', 'traveltime': 'waktu tempuh',
        'studytime': 'waktu belajar', 'failures': 'riwayat tidak naik kelas',
        'schoolsup': 'bimbingan tambahan sekolah', 'famsup': 'dukungan belajar keluarga',
        'paid': 'les berbayar', 'activities': 'ekstrakurikuler', 'nursery': 'riwayat TK',
        'higher': 'keinginan kuliah', 'internet': 'akses internet', 'romantic': 'status berpacaran',
        'famrel': 'kualitas hubungan keluarga', 'freetime': 'waktu luang',
        'goout': 'frekuensi keluar bersama teman', 'Dalc': 'konsumsi alkohol hari kerja',
        'Walc': 'konsumsi alkohol akhir pekan', 'health': 'kondisi kesehatan',
        'absences': 'jumlah ketidakhadiran'}

for nama, df in DATA_MENTAH.items():
    kurang = set(KAMUS.kolom) - set(df.columns)
    assert not kurang, f'{nama}: kolom tidak ditemukan {kurang}'

print(f'{len(FAKTOR)} faktor kandidat: ' +
      ', '.join(f'{t} {int((KAMUS.tipe == t).sum())}' for t in ['biner', 'nominal', 'ordinal', 'numerik']))
display(KAMUS)

In [ ]:
# Gambaran umum, statistik deskriptif, dan komposisi kategori
def ringkasan_awal(df, nama):
    a = CONFIG['AMBANG_LULUS']
    return {'dataset': nama, 'baris': len(df), 'kolom': df.shape[1],
            'missing': int(df.isna().sum().sum()), 'duplikat': int(df.duplicated().sum()),
            'lulus G1': pct((df.G1 >= a).mean()), 'lulus G2': pct((df.G2 >= a).mean()),
            'lulus G3': pct((df.G3 >= a).mean()), 'jumlah G3 = 0': int((df.G3 == 0).sum())}


display(pd.DataFrame([ringkasan_awal(d, n) for n, d in DATA_MENTAH.items()]))

kol_angka = [c for c in KAMUS.kolom if TIPE[c] in ('ordinal', 'numerik', 'nilai')]
for nama, df in DATA_MENTAH.items():
    print(f'\nStatistik deskriptif kolom angka: {nama}')
    display(df[kol_angka].describe().T.round(2))

rows = []
for nama, df in DATA_MENTAH.items():
    for c in [f for f in FAKTOR if TIPE[f] in ('biner', 'nominal')]:
        vc = df[c].value_counts(normalize=True)
        rows.append({'dataset': nama, 'kolom': c, 'komposisi': ', '.join(f'{k} {pct(v)}' for k, v in vc.items())})
print('\nKomposisi kolom kategorikal')
display(pd.DataFrame(rows).pivot(index='kolom', columns='dataset', values='komposisi'))

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey='row')
for i, (nama, df) in enumerate(DATA_MENTAH.items()):
    for j, g in enumerate(['G1', 'G2', 'G3']):
        ax = axes[i, j]
        vc = df[g].value_counts().reindex(range(0, 21), fill_value=0)
        warna = ['#9E9E9E' if v == 0 else (WARNA['Comeback'] if v >= CONFIG['AMBANG_LULUS'] else WARNA['Gagal'])
                 for v in vc.index]
        ax.bar(vc.index, vc.values, color=warna)
        ax.axvline(CONFIG['AMBANG_LULUS'] - 0.5, ls='--', c='k', lw=1)
        ax.set_title(f'{nama}: {g}')
        ax.set_xlabel('Nilai')
        if j == 0:
            ax.set_ylabel('Jumlah siswa')
fig.suptitle('Distribusi nilai sebelum cleaning (abu-abu = nilai 0, merah = gagal, hijau = lulus)', y=1.02)
fig.tight_layout()
simpan(fig, 'eda_distribusi_nilai_mentah')

## Tahap 2. Membersihkan data
**Tujuan:** memastikan setiap baris adalah siswa yang benar-benar menjalani ketiga periode penilaian.

**Alasan:** nilai 0 pada skala 0 sampai 20 diperlakukan sebagai tidak mengikuti penilaian, bukan hasil belajar. Jika dibiarkan, siswa ini otomatis masuk kelompok gagal dan mencemari perbandingan. Baris tersebut diprofilkan dulu, lalu dikeluarkan. Absences yang ekstrem tidak dibuang karena nilainya nyata dan ditangani dengan uji berbasis peringkat.

In [ ]:
ATURAN_RENTANG = {'age': (15, 22), 'Medu': (0, 4), 'Fedu': (0, 4), 'traveltime': (1, 4), 'studytime': (1, 4),
                  'failures': (0, 3), 'famrel': (1, 5), 'freetime': (1, 5), 'goout': (1, 5), 'Dalc': (1, 5),
                  'Walc': (1, 5), 'health': (1, 5), 'absences': (0, 93), 'G1': (0, 20), 'G2': (0, 20),
                  'G3': (0, 20)}
NILAI_SAH = {'school': {'GP', 'MS'}, 'sex': {'F', 'M'}, 'address': {'U', 'R'}, 'famsize': {'LE3', 'GT3'},
             'Pstatus': {'T', 'A'}, 'Mjob': {'teacher', 'health', 'services', 'at_home', 'other'},
             'Fjob': {'teacher', 'health', 'services', 'at_home', 'other'},
             'reason': {'home', 'reputation', 'course', 'other'}, 'guardian': {'mother', 'father', 'other'}}
for f in FAKTOR:
    if TIPE[f] == 'biner' and f not in NILAI_SAH:
        NILAI_SAH[f] = {'yes', 'no'}


def profil_nilai_nol(df):
    g3 = df.G3 == 0
    return {'baris dengan nilai 0 (G1/G2/G3)': int((df[['G1', 'G2', 'G3']] == 0).any(axis=1).sum()),
            'G3 = 0': int(g3.sum()), 'G2 = 0': int((df.G2 == 0).sum()), 'G1 = 0': int((df.G1 == 0).sum()),
            'G3 = 0 dan G2 = 0': int((g3 & (df.G2 == 0)).sum()),
            'absences = 0 pada siswa G3 = 0': pct((df.loc[g3, 'absences'] == 0).mean()) if g3.any() else '-',
            'absences = 0 pada siswa lain': pct((df.loc[~g3, 'absences'] == 0).mean())}


def bersihkan(df, nama):
    log = {'dataset': nama, 'n_awal': len(df)}
    df = df.copy()
    for c in ATURAN_RENTANG:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    kolom = KAMUS.kolom.tolist()
    log['missing'] = int(df[kolom].isna().any(axis=1).sum())
    df = df.dropna(subset=kolom)
    log['duplikat'] = int(df.duplicated().sum())
    df = df.drop_duplicates()
    luar = pd.Series(False, index=df.index)
    for c, (lo, hi) in ATURAN_RENTANG.items():
        luar |= (df[c] < lo) | (df[c] > hi)
    for c, sah in NILAI_SAH.items():
        luar |= ~df[c].isin(sah)
    log['di_luar_rentang'] = int(luar.sum())
    df = df[~luar]
    nol = (df[['G1', 'G2', 'G3']] == 0).any(axis=1)
    log['nilai_0_dibuang'] = int(nol.sum())
    df = df[~nol].copy()
    for c in ATURAN_RENTANG:
        df[c] = df[c].astype(int)
    log['n_akhir'] = len(df)
    log['total_dibuang'] = log['n_awal'] - log['n_akhir']
    return df.reset_index(drop=True), log


print('Profil baris bernilai 0 sebelum dibuang:')
display(pd.DataFrame({n: profil_nilai_nol(d) for n, d in DATA_MENTAH.items()}))

DATA, LOG = {}, []
for nama, df in DATA_MENTAH.items():
    DATA[nama], log = bersihkan(df, nama)
    LOG.append(log)
LOG = pd.DataFrame(LOG)
print('Log cleaning:')
display(LOG)
simpan_tabel(LOG, 'log_cleaning')

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
for ax, (nama, df) in zip(axes, DATA.items()):
    sns.heatmap(df[kol_angka].corr(method='spearman'), ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                annot=True, fmt='.1f', annot_kws={'size': 7}, square=True, cbar=(ax is axes[-1]))
    ax.set_title(f'Korelasi Spearman setelah cleaning: {nama}')
fig.tight_layout()
simpan(fig, 'eda_heatmap_korelasi')

## Tahap 3. Menentukan kategori lintasan nilai
**Tujuan:** memberi label lintasan pada setiap siswa. Variabel hasil analisis adalah **perubahan status**, bukan level nilai akhir.

Dua label dibuat: empat kategori untuk analisis deskriptif dan antar kategori, serta label biner (Comeback vs Tetap Gagal) untuk analisis utama.

In [ ]:
def beri_label(df):
    df = df.copy()
    a = CONFIG['AMBANG_LULUS']
    for g in ['G1', 'G2', 'G3']:
        df[f'lulus_{g}'] = df[g] >= a
    l1, l2, l3 = df.lulus_G1, df.lulus_G2, df.lulus_G3
    kondisi = [~l1 & l2 & l3, ~l1 & ~l2 & l3, ~l1 & l2 & ~l3, ~l1 & ~l2 & ~l3, l1 & l3, l1 & ~l3]
    df['kategori'] = np.select(kondisi, URUTAN_SEMUA, default='?')
    df['pola'] = [f"{'L' if a_ else 'G'}-{'L' if b_ else 'G'}-{'L' if c_ else 'G'}" for a_, b_, c_ in zip(l1, l2, l3)]
    df['selisih_G3_G1'] = df.G3 - df.G1
    return df


HASIL = {}
for nama, df in DATA.items():
    d = beri_label(df)
    fokus = d[~d.lulus_G1].copy()
    kel_gagal = ['Tetap Gagal'] + (['Comeback Sementara'] if CONFIG['SERTAKAN_SEMENTARA_DI_BINER'] else [])
    biner = fokus[fokus.kategori.isin(['Comeback Awal', 'Comeback Akhir'] + kel_gagal)].copy()
    biner['comeback'] = biner.kategori.isin(['Comeback Awal', 'Comeback Akhir']).astype(int)
    HASIL[nama] = {'df': d, 'fokus': fokus, 'biner': biner}

pola = pd.concat({n: H['df'].pola.value_counts() for n, H in HASIL.items()}, axis=1).fillna(0).astype(int)
pola.index.name = 'pola G1-G2-G3 (L = lulus, G = gagal)'
print('Delapan kombinasi status untuk seluruh siswa:')
display(pola.sort_index())

rows = []
for nama, H in HASIL.items():
    vc = H['fokus'].kategori.value_counts().reindex(URUTAN_FOKUS, fill_value=0)
    for k, v in vc.items():
        rows.append({'dataset': nama, 'kategori': k, 'n': int(v), 'persen': v / len(H['fokus'])})
KOMPOSISI = pd.DataFrame(rows)
tampil = KOMPOSISI.assign(persen=KOMPOSISI.persen.map(lambda v: pct(v, 1)))
print('Kategori siswa dengan G1 < 10:')
display(tampil.pivot(index='kategori', columns='dataset', values=['n', 'persen']).reindex(URUTAN_FOKUS))
simpan_tabel(KOMPOSISI, 'komposisi_kategori')

for nama, H in HASIL.items():
    b = H['biner']
    n_sem = int((H['fokus'].kategori == 'Comeback Sementara').sum())
    print(f"{nama}: analisis biner = {int(b.comeback.sum())} Comeback vs {int((b.comeback == 0).sum())} Tetap Gagal; "
          f"Comeback Sementara = {n_sem} siswa "
          f"({'ikut' if n_sem >= CONFIG['MIN_N_KELOMPOK'] else 'tidak ikut'} uji antar kategori)")

## Tahap 4. Memetakan pola transisi nilai
**Tujuan:** memperlihatkan pergerakan siswa antar status sebelum masuk ke faktor.

- **Sankey**: aliran siswa antar status lulus/gagal, lebar pita = jumlah siswa (interaktif, juga tersimpan sebagai HTML).
- **Matriks transisi**: persentase perpindahan status G1 ke G3.
- **Komposisi kategori**: proporsi empat kategori pada siswa G1 < 10 di kedua dataset.
- **Slope chart**: kapan setiap kategori mulai naik.
- **Selisih nilai**: besar perubahan G3 dikurangi G1, bukan hanya status.

In [ ]:
def gambar_sankey(df, nama):
    label = ['G1 Lulus', 'G1 Gagal', 'G2 Lulus', 'G2 Gagal', 'G3 Lulus', 'G3 Gagal']
    idx = {l: i for i, l in enumerate(label)}
    status = lambda s: np.where(s, 'Lulus', 'Gagal')
    src, tgt, val, col = [], [], [], []
    for a, b in [('G1', 'G2'), ('G2', 'G3')]:
        t = pd.crosstab(status(df[f'lulus_{a}']), status(df[f'lulus_{b}']))
        for s_ in t.index:
            for d_ in t.columns:
                if t.loc[s_, d_] > 0:
                    src.append(idx[f'{a} {s_}'])
                    tgt.append(idx[f'{b} {d_}'])
                    val.append(int(t.loc[s_, d_]))
                    col.append('rgba(46,125,50,0.35)' if d_ == 'Lulus' else 'rgba(198,40,40,0.35)')
    n_node = []
    for g in ['G1', 'G2', 'G3']:
        n_node += [int(df[f'lulus_{g}'].sum()), int((~df[f'lulus_{g}']).sum())]
    fig = go.Figure(go.Sankey(
        arrangement='snap',
        node=dict(label=[f'{l} ({n})' for l, n in zip(label, n_node)], pad=22, thickness=18,
                  color=['#2E7D32', '#C62828'] * 3),
        link=dict(source=src, target=tgt, value=val, color=col)))
    fig.update_layout(title=f'Aliran status G1 → G2 → G3: {nama} (n = {len(df)})', font_size=12, height=420)
    fig.write_html(os.path.join(OUT, f'sankey_{KODE[nama]}.html'), include_plotlyjs='cdn')
    fig.show()


for nama, H in HASIL.items():
    gambar_sankey(H['df'], nama)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), gridspec_kw={'width_ratios': [1, 1, 1.7]})
for ax, (nama, H) in zip(axes[:2], HASIL.items()):
    d = H['df']
    s1 = np.where(d.lulus_G1, 'Lulus G1', 'Gagal G1')
    s3 = np.where(d.lulus_G3, 'Lulus G3', 'Gagal G3')
    n = pd.crosstab(s1, s3).reindex(index=['Gagal G1', 'Lulus G1'], columns=['Gagal G3', 'Lulus G3'], fill_value=0)
    p = n.div(n.sum(1), axis=0)
    annot = np.array([[f'{pct(p.iloc[i, j])}\n(n = {n.iloc[i, j]})' for j in range(2)] for i in range(2)])
    sns.heatmap(p, annot=annot, fmt='', cmap='Blues', vmin=0, vmax=1, cbar=False, ax=ax,
                linewidths=1, linecolor='white', annot_kws={'size': 11})
    ax.set_title(f'Matriks transisi G1 → G3: {nama}')
    ax.set_xlabel('')
    ax.set_ylabel('')

ax = axes[2]
komp = KOMPOSISI.pivot(index='dataset', columns='kategori', values='n')[URUTAN_FOKUS]
prop = komp.div(komp.sum(1), axis=0)
kiri = np.zeros(len(prop))
for k in URUTAN_FOKUS:
    ax.barh(prop.index, prop[k], left=kiri, color=WARNA[k], label=k, edgecolor='white')
    for i, (v, l) in enumerate(zip(prop[k], kiri)):
        if v >= 0.05:
            ax.text(l + v / 2, i, f'{pct(v)}\n({komp[k].iloc[i]})', ha='center', va='center', fontsize=9,
                    color='black' if k in ('Comeback Akhir', 'Comeback Sementara') else 'white')
    kiri += prop[k].values
ax.set_xlim(0, 1)
ax.set_title('Komposisi kategori siswa dengan G1 < 10')
ax.legend(ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.12), frameon=False, fontsize=9)
fig.tight_layout()
simpan(fig, 'transisi_matriks_dan_komposisi')

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, (nama, H) in zip(axes, HASIL.items()):
    d = H['df']
    ada = [k for k in URUTAN_SEMUA if (d.kategori == k).any()]
    rata = d.groupby('kategori')[['G1', 'G2', 'G3']].mean().reindex(ada)
    n = d.kategori.value_counts()
    for k, r in rata.iterrows():
        ax.plot(['G1', 'G2', 'G3'], r.values, marker='o', lw=2.5, color=WARNA[k], label=f'{k} (n = {n[k]})')
    ax.axhline(CONFIG['AMBANG_LULUS'], ls='--', c='k', lw=1)
    ax.set_title(f'Rata-rata nilai per kategori: {nama}')
    ax.set_ylabel('Nilai rata-rata')
    ax.legend(fontsize=8, loc='best')
fig.tight_layout()
simpan(fig, 'transisi_slope_chart')

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharey=True)
for ax, (nama, H) in zip(axes, HASIL.items()):
    f_ = H['fokus']
    urut = [k for k in URUTAN_FOKUS if (f_.kategori == k).any()]
    sns.boxplot(data=f_, x='kategori', y='selisih_G3_G1', order=urut, hue='kategori', hue_order=urut,
                palette=WARNA, legend=False, showfliers=False, ax=ax)
    sns.stripplot(data=f_, x='kategori', y='selisih_G3_G1', order=urut, color='k', alpha=0.35, size=3, ax=ax)
    ax.axhline(0, c='k', lw=1)
    ax.set_xticks(range(len(urut)))
    ax.set_xticklabels(label_kategori(f_, urut), fontsize=9)
    ax.set_title(f'Perubahan nilai (G3 dikurangi G1): {nama}')
    ax.set_xlabel('')
    ax.set_ylabel('Selisih nilai')
fig.tight_layout()
simpan(fig, 'transisi_selisih_nilai')

## Tahap 5. Memeriksa kesetaraan titik awal G1
**Tujuan:** memastikan perbedaan faktor nanti tidak sekadar mencerminkan perbedaan G1. Siswa dengan G1 = 9 hanya perlu naik 1 poin untuk lulus, sedangkan siswa dengan G1 = 5 perlu naik 5 poin.

**Metode:** Mann-Whitney U (Comeback vs Tetap Gagal) dan Kruskal-Wallis (antar kategori). Strata G1 dibentuk dari sebaran G1 saja (titik potong yang membagi siswa paling mendekati 50:50), tanpa melihat hasil comeback. Strata ini dipakai di Tahap 8.

In [ ]:
def pilih_batas_strata(g1):
    kandidat = sorted(g1.unique())[:-1]
    return min(kandidat, key=lambda s: abs((g1 <= s).mean() - 0.5))


rows = []
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for ax, (nama, H) in zip(axes, HASIL.items()):
    b, f_ = H['biner'], H['fokus']
    x1, x0 = b.loc[b.comeback == 1, 'G1'], b.loc[b.comeback == 0, 'G1']
    U, p = stats.mannwhitneyu(x1, x0, alternative='two-sided')
    r = 2 * U / (len(x1) * len(x0)) - 1
    kat_uji = [k for k in URUTAN_FOKUS if (f_.kategori == k).sum() >= CONFIG['MIN_N_KELOMPOK']]
    _, p_kw = stats.kruskal(*[f_.loc[f_.kategori == k, 'G1'] for k in kat_uji])
    batas = pilih_batas_strata(b.G1)
    H['batas_strata'] = batas
    H['g1_awal'] = dict(med1=x1.median(), med0=x0.median(), mean1=x1.mean(), mean0=x0.mean(), p=p, r=r, p_kw=p_kw)
    rows.append({'dataset': nama, 'median G1 Comeback': x1.median(), 'median G1 Tetap Gagal': x0.median(),
                 'rerata G1 Comeback': round(x1.mean(), 2), 'rerata G1 Tetap Gagal': round(x0.mean(), 2),
                 'p Mann-Whitney': round(p, 4), 'rank-biserial r': round(r, 3),
                 'p Kruskal-Wallis antar kategori': round(p_kw, 4),
                 'strata G1': f'rendah: G1 <= {batas} (n = {(b.G1 <= batas).sum()}), '
                              f'dekat ambang: G1 > {batas} (n = {(b.G1 > batas).sum()})'})
    urut = [k for k in URUTAN_FOKUS if (f_.kategori == k).any()]
    sns.boxplot(data=f_, x='kategori', y='G1', order=urut, hue='kategori', hue_order=urut, palette=WARNA,
                legend=False, showfliers=False, ax=ax)
    sns.stripplot(data=f_, x='kategori', y='G1', order=urut, color='k', alpha=0.35, size=3, jitter=0.25, ax=ax)
    ax.set_xticks(range(len(urut)))
    ax.set_xticklabels(label_kategori(f_, urut), fontsize=9)
    ax.set_title(f'Nilai G1 per kategori: {nama}')
    ax.set_xlabel('')
display(pd.DataFrame(rows))
fig.tight_layout()
simpan(fig, 'g1_titik_awal')

## Tahap 6. Menyiapkan faktor
**Tujuan:** menyiapkan semua faktor agar diuji dengan prosedur yang sama. Faktor **tidak diseleksi berdasarkan hasil** sebelum diuji.

Penyaringan yang dilakukan hanya berdasarkan frekuensi pada siswa G1 < 10:
- Kategori nominal dengan jumlah siswa kurang dari `MIN_N_KATEGORI` digabung ke `other`.
- Faktor yang satu nilainya mendominasi lebih dari `BATAS_DOMINAN` dikeluarkan.
- Faktor biner dikodekan 1 untuk level positif (misalnya `yes`, `U`, `M`) dan 0 untuk sebaliknya.

In [ ]:
def siapkan_faktor(fokus):
    X = pd.DataFrame(index=fokus.index)
    aktif, catatan = [], []
    for f in FAKTOR:
        s = fokus[f].copy()
        if TIPE[f] == 'nominal':
            vc = s.value_counts()
            jarang = vc[vc < CONFIG['MIN_N_KATEGORI']].index.tolist()
            if jarang:
                s = s.where(~s.isin(jarang), 'other')
                catatan.append(f'{f}: kategori {jarang} digabung ke "other" (n < {CONFIG["MIN_N_KATEGORI"]})')
        dominan = s.value_counts(normalize=True).iloc[0]
        if s.nunique() < 2 or dominan > CONFIG['BATAS_DOMINAN']:
            catatan.append(f'{f}: dikeluarkan karena satu nilai mendominasi ({pct(dominan)})')
            continue
        X[f] = (s == LEVEL_POSITIF[f]).astype(int) if TIPE[f] == 'biner' else s
        aktif.append(f)
    return X, aktif, catatan


for nama, H in HASIL.items():
    X, aktif, catatan = siapkan_faktor(H['fokus'])
    H['X_fokus'], H['aktif'] = X, aktif
    H['X'] = X.loc[H['biner'].index]
    H['y'] = H['biner']['comeback']
    print(f'{nama}: {len(aktif)} faktor aktif')
    for c in catatan:
        print('   -', c)

## Tahap 7. Uji statistik faktor: Comeback vs Tetap Gagal
**Tujuan:** mengetahui faktor mana yang berbeda dan seberapa besar perbedaannya.

| Tipe faktor | Uji (apakah berbeda) | Ukuran efek (seberapa berbeda) |
|---|---|---|
| Ordinal dan numerik | Mann-Whitney U | Rank-biserial r (bertanda) |
| Biner | Fisher exact | Phi (bertanda) |
| Nominal | Chi-square, atau chi-square permutasi jika frekuensi harapan < 5 | Cramér's V terkoreksi bias |

Tanda positif berarti nilai faktor (atau level positif) lebih tinggi/lebih sering pada siswa Comeback. Patokan efek: sekitar 0,1 kecil, 0,3 sedang, 0,5 besar. Semua p-value dikoreksi dengan **Benjamini-Hochberg** (q-value) karena sekitar 30 uji dilakukan sekaligus.

Status faktor: **Lolos** jika q < ALPHA dan |efek| ≥ EFEK_MIN, **Indikasi** jika q < ALPHA_INDIKASI dan |efek| ≥ EFEK_MIN.

In [ ]:
def chi2_stat(t):
    t = np.asarray(t, dtype=float)
    e = t.sum(1, keepdims=True) * t.sum(0, keepdims=True) / t.sum()
    with np.errstate(divide='ignore', invalid='ignore'):
        return float(np.where(e > 0, (t - e) ** 2 / e, 0.0).sum())


def cramers_v_koreksi(chi2, n, r, k):
    # Cramer's V dengan koreksi bias untuk sampel kecil (Bergsma, 2013)
    phi2 = max(0.0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rk = r - (r - 1) ** 2 / (n - 1)
    kk = k - (k - 1) ** 2 / (n - 1)
    d = min(rk - 1, kk - 1)
    return float(np.sqrt(phi2 / d)) if d > 0 else np.nan


def kode_kategori(x):
    return np.unique(np.asarray(x).astype(str), return_inverse=True)[1].ravel()


def hitung_efek(x, y, tipe):
    x = np.asarray(x)
    y = np.asarray(y).astype(int)
    n1 = int(y.sum())
    n0 = len(y) - n1
    if n1 < 2 or n0 < 2:
        return np.nan
    if tipe in ('ordinal', 'numerik'):
        rk = stats.rankdata(x.astype(float))
        U1 = rk[y == 1].sum() - n1 * (n1 + 1) / 2
        return 2 * U1 / (n1 * n0) - 1
    if tipe == 'biner':
        x = x.astype(float)
        return np.nan if x.std() == 0 else float(np.corrcoef(x, y)[0, 1])
    kode = kode_kategori(x)
    k = kode.max() + 1
    if k < 2:
        return np.nan
    t = np.bincount(kode * 2 + y, minlength=2 * k).reshape(k, 2)
    return cramers_v_koreksi(chi2_stat(t), len(y), k, 2)


def uji_kontingensi(x, g, rng):
    xk, gk = kode_kategori(x), kode_kategori(g)
    kx, kg = xk.max() + 1, gk.max() + 1
    t = np.bincount(xk * kg + gk, minlength=kx * kg).reshape(kx, kg)
    chi2 = chi2_stat(t)
    e = t.sum(1, keepdims=True) * t.sum(0, keepdims=True) / t.sum()
    if (e < 5).any():
        hit = 0
        for _ in range(CONFIG['N_PERMUTASI']):
            tp = np.bincount(xk * kg + rng.permutation(gk), minlength=kx * kg).reshape(kx, kg)
            hit += chi2_stat(tp) >= chi2 - 1e-9
        return chi2, (hit + 1) / (CONFIG['N_PERMUTASI'] + 1), 'Chi-square permutasi', t
    return chi2, float(stats.chi2.sf(chi2, (kx - 1) * (kg - 1))), 'Chi-square', t


def label_efek(e):
    a = abs(e)
    if pd.isna(a):
        return '-'
    return 'besar' if a >= 0.5 else 'sedang' if a >= 0.3 else 'kecil' if a >= 0.1 else 'sangat kecil'


def arah_teks(f, tipe, efek):
    if pd.isna(efek):
        return '-'
    if tipe in ('ordinal', 'numerik'):
        return 'lebih tinggi pada Comeback' if efek > 0 else 'lebih rendah pada Comeback'
    if tipe == 'biner':
        return f'{f} = {LEVEL_POSITIF[f]} ' + ('lebih sering pada Comeback' if efek > 0 else 'lebih jarang pada Comeback')
    return 'distribusi kategori berbeda (lihat ringkasan)'


def tentukan_status(q, efek_abs):
    return np.select([(q < A) & (efek_abs >= E), (q < AI) & (efek_abs >= E)], ['Lolos', 'Indikasi'], 'Tidak bermakna')


def uji_semua_faktor(X, y, aktif, rng):
    y = pd.Series(np.asarray(y).astype(int), index=X.index)
    baris = []
    for f in aktif:
        tipe, x = TIPE[f], X[f]
        x1, x0 = x[y == 1], x[y == 0]
        if tipe in ('ordinal', 'numerik'):
            stat, p = stats.mannwhitneyu(x1.astype(float), x0.astype(float), alternative='two-sided')
            uji, jenis = 'Mann-Whitney U', 'rank-biserial r'
            ringkas = f'median {x1.median():g} vs {x0.median():g}; rerata {x1.mean():.2f} vs {x0.mean():.2f}'
        elif tipe == 'biner':
            tab = [[int(((x == 1) & (y == 1)).sum()), int(((x == 1) & (y == 0)).sum())],
                   [int(((x == 0) & (y == 1)).sum()), int(((x == 0) & (y == 0)).sum())]]
            stat, p = stats.fisher_exact(tab)
            uji, jenis = 'Fisher exact', 'phi'
            ringkas = (f'tingkat comeback {f} = {LEVEL_POSITIF[f]}: {pct(y[x == 1].mean())} '
                       f'vs {LEVEL_NEGATIF[f]}: {pct(y[x == 0].mean())}')
        else:
            stat, p, uji, _ = uji_kontingensi(x, y, rng)
            jenis = "Cramer's V terkoreksi"
            laju = y.groupby(x).mean().sort_values(ascending=False)
            ringkas = 'tingkat comeback: ' + ', '.join(f'{k} {pct(v)}' for k, v in laju.items())
        baris.append(dict(faktor=f, tipe=tipe, peran=PERAN[f], uji=uji, statistik=stat, p=p,
                          efek=hitung_efek(x, y, tipe), jenis_efek=jenis, ringkas=ringkas))
    T = pd.DataFrame(baris)
    T['q'] = multipletests(T['p'], method='fdr_bh')[1]
    T['efek_abs'] = T['efek'].abs()
    T['besar_efek'] = T['efek'].map(label_efek)
    T['arah'] = [arah_teks(f, t, e) for f, t, e in zip(T.faktor, T.tipe, T.efek)]
    T['status'] = tentukan_status(T.q, T.efek_abs)
    return T.sort_values(['q', 'efek_abs'], ascending=[True, False]).reset_index(drop=True)


for nama, H in HASIL.items():
    T = uji_semua_faktor(H['X'], H['y'], H['aktif'], RNG)
    H['uji'] = T
    simpan_tabel(T, f'uji_faktor_{KODE[nama]}')
    print(f'\n=== {nama}: {(T.status == "Lolos").sum()} faktor lolos, {(T.status == "Indikasi").sum()} indikasi '
          f'dari {len(T)} faktor ===')
    kolom = ['faktor', 'tipe', 'uji', 'p', 'q', 'efek', 'jenis_efek', 'besar_efek', 'arah', 'status', 'ringkas']
    display(T[kolom].round({'p': 4, 'q': 4, 'efek': 3}))

## Tahap 8. Menguji validitas temuan
**Tujuan:** memastikan faktor bukan hasil kebetulan, bawaan nilai awal, atau kebetulan sampel.

| Pemeriksaan | Metode | Kriteria |
|---|---|---|
| Kontrol titik awal (strata) | Efek dihitung ulang di strata G1 rendah dan dekat ambang | Arah sama dan efek minimal EFEK_REPLIKASI di kedua strata |
| Kontrol titik awal (model) | Regresi logistik `comeback ~ G1 + faktor`, uji likelihood ratio, koreksi BH | q < ALPHA dan arah odds ratio sama dengan arah efek |
| Stabilitas | Bootstrap bertingkat (resampling per kelompok) sebanyak N_BOOTSTRAP kali | Proporsi resample dengan arah sama dan efek ≥ EFEK_MIN minimal STABILITAS_MIN |
| Korelasi antar faktor | Spearman antar faktor kunci | Pasangan dengan korelasi ≥ 0,5 dibaca sebagai satu temuan gabungan |

In [ ]:
def kodekan_logit(X, f):
    tipe = TIPE[f]
    if tipe == 'nominal':
        urutan = X[f].value_counts().index.tolist()
        D = pd.get_dummies(pd.Categorical(X[f], categories=urutan), prefix=f, drop_first=True, dtype=float)
        D.index = X.index
        return D, f'dummy, referensi {urutan[0]}'
    if tipe == 'numerik':
        return ((X[f] - X[f].mean()) / X[f].std()).astype(float).to_frame(f), 'per 1 SD'
    if tipe == 'ordinal':
        return X[f].astype(float).to_frame(f), 'per 1 tingkat'
    return X[f].astype(float).to_frame(f), f'{LEVEL_POSITIF[f]} vs {LEVEL_NEGATIF[f]}'


def logit_kontrol_g1(X, y, g1, aktif):
    y = np.asarray(y).astype(int)
    dasar = pd.DataFrame({'const': 1.0, 'G1': np.asarray(g1, dtype=float)}, index=X.index)
    m0 = sm.Logit(y, dasar).fit(disp=0)
    baris = []
    for f in aktif:
        D, satuan = kodekan_logit(X, f)
        rec = dict(faktor=f, satuan_OR=satuan, OR_kontrol_G1=np.nan, CI_bawah=np.nan, CI_atas=np.nan,
                   p_LR=np.nan, catatan='')
        try:
            m1 = sm.Logit(y, pd.concat([dasar, D], axis=1)).fit(disp=0, maxiter=200)
            lr = max(0.0, 2 * (m1.llf - m0.llf))
            rec['p_LR'] = float(stats.chi2.sf(lr, D.shape[1]))
            if D.shape[1] == 1:
                b, ci = m1.params[f], m1.conf_int().loc[f]
                rec.update(OR_kontrol_G1=float(np.exp(b)), CI_bawah=float(np.exp(ci.iloc[0])),
                           CI_atas=float(np.exp(ci.iloc[1])))
                if abs(b) > 8:
                    rec['catatan'] = 'koefisien ekstrem, kemungkinan separasi'
            if not m1.mle_retvals.get('converged', True):
                rec['catatan'] = 'tidak konvergen'
        except Exception as e:
            rec['catatan'] = f'gagal: {type(e).__name__}'
        baris.append(rec)
    T = pd.DataFrame(baris)
    T['q_LR'] = np.nan
    ok = T['p_LR'].notna()
    if ok.any():
        T.loc[ok, 'q_LR'] = multipletests(T.loc[ok, 'p_LR'], method='fdr_bh')[1]
    return T


def efek_per_strata(X, y, g1, aktif, batas):
    y = np.asarray(y).astype(int)
    rendah = np.asarray(g1) <= batas
    rows = {f: [hitung_efek(X[f].to_numpy()[rendah], y[rendah], TIPE[f]),
                hitung_efek(X[f].to_numpy()[~rendah], y[~rendah], TIPE[f])] for f in aktif}
    return pd.DataFrame(rows, index=['efek_G1_rendah', 'efek_G1_dekat']).T


def bootstrap_efek(X, y, aktif, efek_penuh, rng, B):
    y = np.asarray(y).astype(int)
    i1, i0 = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    arr = {f: (kode_kategori(X[f]) if TIPE[f] == 'nominal' else X[f].to_numpy()) for f in aktif}
    hasil = np.full((B, len(aktif)), np.nan)
    for b in range(B):
        s = np.concatenate([rng.choice(i1, len(i1)), rng.choice(i0, len(i0))])
        yb = y[s]
        for j, f in enumerate(aktif):
            hasil[b, j] = hitung_efek(arr[f][s], yb, TIPE[f])
    rows = []
    for j, f in enumerate(aktif):
        e = hasil[:, j]
        e = e[~np.isnan(e)]
        ef = efek_penuh[f]
        if TIPE[f] == 'nominal':
            stabil = np.mean(e >= E)
        else:
            stabil = np.mean((np.sign(e) == np.sign(ef)) & (np.abs(e) >= E))
        rows.append(dict(faktor=f, CI95_bawah=np.percentile(e, 2.5), CI95_atas=np.percentile(e, 97.5),
                         stabilitas=stabil))
    return pd.DataFrame(rows)


for nama, H in HASIL.items():
    g1 = H['biner']['G1']
    H['strata'] = efek_per_strata(H['X'], H['y'], g1, H['aktif'], H['batas_strata'])
    H['logit'] = logit_kontrol_g1(H['X'], H['y'], g1, H['aktif'])
    efek_penuh = H['uji'].set_index('faktor')['efek']
    H['boot'] = bootstrap_efek(H['X'], H['y'], H['aktif'], efek_penuh, RNG, CONFIG['N_BOOTSTRAP'])
    print(f'{nama}: strata, regresi logistik terkontrol G1, dan bootstrap ({CONFIG["N_BOOTSTRAP"]}x) selesai')

fig, axes = plt.subplots(1, 2, figsize=(17, 10))
for ax, (nama, H) in zip(axes, HASIL.items()):
    T = H['uji'].merge(H['boot'], on='faktor').sort_values('efek').reset_index(drop=True)
    ypos = np.arange(len(T))
    warna = T.status.map(WARNA_STATUS)
    ax.hlines(ypos, T.CI95_bawah, T.CI95_atas, color=warna, lw=2.2)
    for yy, e, c, t in zip(ypos, T.efek, warna, T.tipe):
        ax.plot(e, yy, 's' if t == 'nominal' else 'o', color=c, ms=7)
    ax.set_yticks(ypos)
    ax.set_yticklabels(T.faktor)
    for v in (-E, E):
        ax.axvline(v, ls='--', c='grey', lw=0.9)
    ax.axvline(0, c='k', lw=1)
    ax.set_title(f'Ukuran efek dan CI bootstrap 95%: {nama}')
    ax.set_xlabel('Efek (+ = lebih tinggi / lebih sering pada Comeback)')
legenda = [Line2D([0], [0], color=c, marker='o', lw=2, label=s) for s, c in WARNA_STATUS.items()]
legenda.append(Line2D([0], [0], color='k', marker='s', lw=0, label="kotak = Cramer's V (tanpa arah)"))
fig.legend(handles=legenda, loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.03))
fig.tight_layout()
simpan(fig, 'validitas_forest_plot')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, (nama, H) in zip(axes, HASIL.items()):
    T = H['uji']
    kunci = T[(T.status == 'Lolos') & (T.tipe != 'nominal')].faktor.tolist()
    judul = 'faktor kunci'
    if len(kunci) < 2:
        kunci = T[T.tipe != 'nominal'].nlargest(8, 'efek_abs').faktor.tolist()
        judul = '8 faktor dengan efek terbesar (eksploratif)'
    C = H['X'][kunci].astype(float).corr(method='spearman')
    pasangan = [(a, b, C.loc[a, b]) for a, b in combinations(kunci, 2) if abs(C.loc[a, b]) >= 0.5]
    H['pasangan_kuat'] = pasangan
    sns.heatmap(C, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1, square=True, cbar=False, ax=ax)
    ax.set_title(f'Korelasi Spearman antar {judul}: {nama}', fontsize=11)
    print(f'{nama}: pasangan dengan |rho| >= 0,5 =',
          [(a, b, round(v, 2)) for a, b, v in pasangan] if pasangan else 'tidak ada')
fig.tight_layout()
simpan(fig, 'validitas_korelasi_faktor')

## Tahap 9. Matriks konsistensi dan faktor kunci
**Inovasi utama:** setiap faktor dinilai pada enam lapis pemeriksaan. Skor konsistensi (0 sampai 6) menunjukkan seberapa bisa dipercaya sebuah temuan, bukan hanya apakah p-value-nya kecil.

1. Signifikan (q < ALPHA)
2. Efek ≥ EFEK_MIN
3. Tahan kontrol G1 (regresi logistik)
4. Konsisten antar strata G1
5. Stabil di bootstrap
6. Arah sama di dataset lain (replikasi Matematika dan Portugis)

In [ ]:
LAPIS = [f'Signifikan (q < {A})', f'Efek >= {E}', 'Tahan kontrol G1', 'Konsisten antar strata G1',
         f'Stabil bootstrap (>= {int(CONFIG["STABILITAS_MIN"] * 100)}%)', 'Arah sama di dataset lain']


def matriks_konsistensi(nama):
    H, O = HASIL[nama], HASIL[LAIN[nama]]
    T = H['uji'].set_index('faktor')
    L = H['logit'].set_index('faktor').reindex(T.index)
    S = H['strata'].reindex(T.index)
    B = H['boot'].set_index('faktor').reindex(T.index)
    eo = O['uji'].set_index('faktor')['efek'].reindex(T.index)
    bertanda = (T.tipe != 'nominal').to_numpy()
    tanda = np.sign(T.efek.to_numpy())
    r_min = CONFIG['EFEK_REPLIKASI']
    M = pd.DataFrame(index=T.index)
    M[LAPIS[0]] = (T.q < A).to_numpy()
    M[LAPIS[1]] = (T.efek_abs >= E).to_numpy()
    arah_or = np.where(bertanda, np.sign(np.log(L.OR_kontrol_G1.to_numpy())) == tanda, True)
    M[LAPIS[2]] = (L.q_LR.to_numpy() < A) & arah_or
    sr, sd = S.efek_G1_rendah.to_numpy(), S.efek_G1_dekat.to_numpy()
    searah_r = (np.sign(sr) == tanda) & (np.abs(sr) >= r_min)
    searah_d = (np.sign(sd) == tanda) & (np.abs(sd) >= r_min)
    M[LAPIS[3]] = np.where(bertanda, searah_r & searah_d, (sr >= r_min) & (sd >= r_min))
    M[LAPIS[4]] = (B.stabilitas.to_numpy() >= CONFIG['STABILITAS_MIN'])
    eo = eo.to_numpy()
    M[LAPIS[5]] = np.where(bertanda, (np.sign(eo) == tanda) & (np.abs(eo) >= r_min), eo >= r_min)
    return M.fillna(False).astype(bool)


for nama, H in HASIL.items():
    M = matriks_konsistensi(nama)
    H['konsistensi'] = M
    R = (H['uji']
         .merge(H['boot'], on='faktor')
         .merge(H['logit'][['faktor', 'OR_kontrol_G1', 'CI_bawah', 'CI_atas', 'satuan_OR', 'q_LR']], on='faktor')
         .merge(H['strata'], left_on='faktor', right_index=True))
    R['skor_konsistensi'] = R.faktor.map(M.sum(axis=1)).astype(int)
    H['ringkas'] = R.sort_values(['skor_konsistensi', 'efek_abs'], ascending=False).reset_index(drop=True)
    simpan_tabel(M.reset_index().rename(columns={'index': 'faktor'}), f'matriks_konsistensi_{KODE[nama]}')

fig, axes = plt.subplots(1, 2, figsize=(19, 12))
for ax, (nama, H) in zip(axes, HASIL.items()):
    R = H['ringkas'].set_index('faktor')
    Mv = H['konsistensi'].loc[R.index]
    sns.heatmap(Mv.astype(int), ax=ax, cmap=ListedColormap(['#EFEFEF', '#2E7D32']), vmin=0, vmax=1, cbar=False,
                linewidths=0.8, linecolor='white', annot=np.where(Mv.values, '✓', ''), fmt='',
                annot_kws={'color': 'white', 'size': 12, 'weight': 'bold'})
    ax.set_yticks(np.arange(len(R)) + 0.5)
    ax.set_yticklabels([f'{f}  ({R.loc[f, "skor_konsistensi"]}/6)' for f in R.index], rotation=0)
    ax.set_xticklabels(Mv.columns, rotation=30, ha='right')
    ax.set_title(f'Matriks konsistensi faktor: {nama}')
    ax.set_ylabel('')
fig.tight_layout()
simpan(fig, 'matriks_konsistensi')

for nama, H in HASIL.items():
    R = H['ringkas']
    print(f'\n=== {nama}: faktor dengan skor konsistensi tertinggi ===')
    kolom = ['faktor', 'tipe', 'peran', 'efek', 'CI95_bawah', 'CI95_atas', 'q', 'OR_kontrol_G1', 'q_LR',
             'stabilitas', 'skor_konsistensi', 'status']
    display(R[kolom].head(12).round(3))

In [ ]:
def tingkat_per_level(x, y, f):
    tipe = TIPE[f]
    y = np.asarray(y).astype(int)
    if tipe == 'numerik':
        kel = pd.qcut(x.astype(float), q=4, duplicates='drop')
        lab = {iv: (f'{int(np.floor(iv.left)) + 1}' if int(np.floor(iv.left)) + 1 == int(np.floor(iv.right))
                    else f'{int(np.floor(iv.left)) + 1}-{int(np.floor(iv.right))}') for iv in kel.cat.categories}
        level = kel.map(lab).astype(str).to_numpy()
        urutan = [lab[iv] for iv in kel.cat.categories]
    elif tipe == 'biner':
        level = np.where(np.asarray(x) == 1, LEVEL_POSITIF[f], LEVEL_NEGATIF[f])
        urutan = [LEVEL_NEGATIF[f], LEVEL_POSITIF[f]]
    elif tipe == 'ordinal':
        level = np.asarray(x).astype(int).astype(str)
        urutan = sorted(set(level), key=int)
    else:
        level = np.asarray(x).astype(str)
        urutan = sorted(set(level))
    rows = []
    for lv in urutan:
        m = level == lv
        k, n = int(y[m].sum()), int(m.sum())
        if n == 0:
            continue
        lo, hi = proportion_confint(k, n, method='wilson')
        rows.append(dict(level=lv, n=n, laju=k / n, lo=lo, hi=hi))
    return pd.DataFrame(rows)


for nama, H in HASIL.items():
    R = H['ringkas'].head(4)
    fig, axes = plt.subplots(1, len(R), figsize=(4.6 * len(R), 4.2), sharey=True)
    axes = np.atleast_1d(axes)
    laju_umum = H['y'].mean()
    for ax, (_, r) in zip(axes, R.iterrows()):
        D = tingkat_per_level(H['X'][r.faktor], H['y'], r.faktor)
        ax.bar(D.level, D.laju, color=WARNA_STATUS[r.status], alpha=0.85,
               yerr=[np.clip(D.laju - D.lo, 0, None), np.clip(D.hi - D.laju, 0, None)], capsize=4, ecolor='#555')
        for i, (v, n) in enumerate(zip(D.laju, D.n)):
            ax.text(i, 0.02, f'n={n}', ha='center', va='bottom', fontsize=8,
                    color='white' if v > 0.1 else 'black')
        ax.axhline(laju_umum, ls='--', c='k', lw=1)
        ax.set_ylim(0, 1)
        ax.set_title(f'{r.faktor} (efek {angka(r.efek)}, skor {r.skor_konsistensi}/6)', fontsize=10)
        ax.set_xlabel(NAMA.get(r.faktor, r.faktor))
        ax.tick_params(axis='x', labelsize=8)
    axes[0].set_ylabel('Proporsi Comeback (CI Wilson 95%)')
    fig.suptitle(f'Tingkat comeback per level faktor teratas: {nama} (garis putus = rata-rata)', y=1.03)
    fig.tight_layout()
    simpan(fig, f'faktor_tingkat_comeback_{KODE[nama]}')

## Tahap 10. Analisis antar kategori dan G2 sebagai checkpoint
**Tujuan:** melihat apakah faktor kunci membedakan Comeback Awal, Comeback Akhir, Comeback Sementara, dan Tetap Gagal, serta menilai G2 sebagai sinyal peringatan dini.

- Ordinal dan numerik: Kruskal-Wallis + epsilon kuadrat, post-hoc **Dunn** dengan koreksi BH.
- Biner dan nominal: chi-square (permutasi jika sel kecil) + Cramér's V, post-hoc Fisher berpasangan untuk faktor biner.
- Kategori dengan jumlah siswa di bawah `MIN_N_KELOMPOK` (biasanya Comeback Sementara) tetap tampil di grafik, tetapi tidak ikut uji statistik karena terlalu kecil. Turunkan `MIN_N_KELOMPOK` di konfigurasi jika ingin tetap mengujinya, dengan risiko hasil tidak stabil.
- G2 **tidak** masuk daftar faktor karena ikut membentuk label kategori, jadi dianalisis terpisah sebagai checkpoint.

In [ ]:
def uji_dunn(x, g, urutan):
    x = np.asarray(x, dtype=float)
    g = np.asarray(g)
    N = len(x)
    rk = stats.rankdata(x)
    _, cnt = np.unique(x, return_counts=True)
    var_dasar = N * (N + 1) / 12 - (cnt ** 3 - cnt).sum() / (12 * (N - 1))
    rows = []
    for a, b in combinations(urutan, 2):
        na, nb = (g == a).sum(), (g == b).sum()
        z = (rk[g == a].mean() - rk[g == b].mean()) / np.sqrt(var_dasar * (1 / na + 1 / nb))
        rows.append(dict(pasangan=f'{a} vs {b}', z=z, p=2 * stats.norm.sf(abs(z))))
    D = pd.DataFrame(rows)
    D['q'] = multipletests(D.p, method='fdr_bh')[1]
    return D


def uji_antar_kategori(H, faktor_list, rng):
    f_, Xf = H['fokus'], H['X_fokus']
    kat = [k for k in URUTAN_FOKUS if (f_.kategori == k).sum() >= CONFIG['MIN_N_KELOMPOK']]
    m = f_.kategori.isin(kat).to_numpy()
    g = f_.loc[m, 'kategori'].to_numpy()
    rows, posthoc = [], []
    for f in faktor_list:
        x = Xf.loc[m, f]
        tipe = TIPE[f]
        if tipe in ('ordinal', 'numerik'):
            Hs, p = stats.kruskal(*[x[g == k].astype(float) for k in kat])
            ef, jenis, uji = Hs / (len(x) - 1), 'epsilon kuadrat', 'Kruskal-Wallis'
            D = uji_dunn(x.to_numpy(), g, kat)
            D.insert(0, 'faktor', f)
            posthoc.append(D)
            ringkas = '; '.join(f'{k}: median {np.median(x[g == k]):g}' for k in kat)
        else:
            chi2, p, uji, t = uji_kontingensi(x.to_numpy(), g, rng)
            ef, jenis = cramers_v_koreksi(chi2, len(x), t.shape[0], t.shape[1]), "Cramer's V"
            if tipe == 'biner':
                ringkas = '; '.join(f'{k}: {pct(x[g == k].mean())} {LEVEL_POSITIF[f]}' for k in kat)
                pr = []
                for a, b in combinations(kat, 2):
                    tab = [[int((x[g == a] == 1).sum()), int((x[g == a] == 0).sum())],
                           [int((x[g == b] == 1).sum()), int((x[g == b] == 0).sum())]]
                    pr.append(dict(faktor=f, pasangan=f'{a} vs {b}', z=np.nan, p=stats.fisher_exact(tab)[1]))
                D = pd.DataFrame(pr)
                D['q'] = multipletests(D.p, method='fdr_bh')[1]
                posthoc.append(D)
            else:
                ringkas = 'lihat tabel kontingensi'
        rows.append(dict(faktor=f, tipe=tipe, uji=uji, p=p, efek=ef, jenis_efek=jenis, ringkas=ringkas))
    T = pd.DataFrame(rows)
    if len(T):
        T['q'] = multipletests(T.p, method='fdr_bh')[1]
    P = pd.concat(posthoc, ignore_index=True) if posthoc else pd.DataFrame()
    return T, P, kat


for nama, H in HASIL.items():
    R = H['ringkas']
    kunci = R[R.status == 'Lolos'].faktor.tolist()
    H['antar_eksploratif'] = not kunci
    daftar_uji = kunci if kunci else R.head(5).faktor.tolist()
    T, P, kat = uji_antar_kategori(H, daftar_uji, RNG)
    H['antar_kategori'], H['posthoc'], H['kat_uji'] = T, P, kat
    label = 'faktor kunci' if kunci else '5 faktor teratas (eksploratif, tidak ada faktor yang lolos)'
    print(f'\n=== {nama}: uji antar kategori untuk {label} ===')
    print('Kategori yang diuji:', ', '.join(kat))
    display(T.round(4))
    if len(P):
        print('Post-hoc berpasangan (q setelah koreksi BH):')
        display(P.round(4))
    simpan_tabel(T, f'antar_kategori_{KODE[nama]}')

for nama, H in HASIL.items():
    f_, Xf = H['fokus'], H['X_fokus']
    pilih = [f for f in H['antar_kategori'].faktor if TIPE[f] != 'nominal'][:4]
    if not pilih:
        continue
    urut = [k for k in URUTAN_FOKUS if (f_.kategori == k).any()]
    fig, axes = plt.subplots(1, len(pilih), figsize=(4.8 * len(pilih), 4.3))
    axes = np.atleast_1d(axes)
    for ax, f in zip(axes, pilih):
        d = pd.DataFrame({'kategori': f_.kategori, 'nilai': Xf[f].astype(float)})
        if TIPE[f] == 'biner':
            prop = d.groupby('kategori')['nilai'].mean().reindex(urut)
            ax.bar(urut, prop.values, color=[WARNA[k] for k in urut])
            ax.set_ylabel(f'Proporsi {f} = {LEVEL_POSITIF[f]}')
            ax.set_ylim(0, 1)
        else:
            sns.boxplot(data=d, x='kategori', y='nilai', order=urut, hue='kategori', hue_order=urut,
                        palette=WARNA, legend=False, showfliers=False, ax=ax)
            sns.stripplot(data=d, x='kategori', y='nilai', order=urut, color='k', alpha=0.3, size=3, ax=ax)
            ax.set_ylabel(f)
        ax.set_title(f'{f}: {NAMA.get(f, f)}', fontsize=10)
        ax.set_xlabel('')
        ax.set_xticks(range(len(urut)))
        ax.set_xticklabels(label_kategori(f_, urut), fontsize=8, rotation=20)
    fig.suptitle(f'Faktor teratas per kategori: {nama}', y=1.02)
    fig.tight_layout()
    simpan(fig, f'antar_kategori_{KODE[nama]}')

In [ ]:
rows = []
for nama, H in HASIL.items():
    f_ = H['fokus']
    tab = []
    for lulus_g2 in [True, False]:
        sub = f_[f_.lulus_G2 == lulus_g2]
        k, n = int(sub.lulus_G3.sum()), len(sub)
        lo, hi = proportion_confint(k, n, method='wilson') if n else (np.nan, np.nan)
        rows.append({'dataset': nama, 'status G2': 'Lulus G2' if lulus_g2 else 'Gagal G2', 'n': n, 'lulus G3': k,
                     'peluang lulus G3': k / n if n else np.nan, 'CI bawah': lo, 'CI atas': hi})
        tab.append([k, n - k])
    H['g2_p'] = stats.fisher_exact(tab)[1] if min(sum(t) for t in tab) > 0 else np.nan
G2 = pd.DataFrame(rows)
for nama, H in HASIL.items():
    H['g2'] = G2[G2.dataset == nama].set_index('status G2')
print('Peluang lulus G3 bagi siswa yang gagal di G1, berdasarkan status G2:')
display(G2.round(3))
simpan_tabel(G2, 'checkpoint_g2')

fig, ax = plt.subplots(figsize=(8, 4.5))
lebar = 0.36
for i, nama in enumerate(HASIL):
    d = G2[G2.dataset == nama]
    xpos = np.arange(2) + (i - 0.5) * lebar
    ax.bar(xpos, d['peluang lulus G3'], width=lebar, color=WARNA[nama], label=nama,
           yerr=[np.clip(d['peluang lulus G3'] - d['CI bawah'], 0, None), np.clip(d['CI atas'] - d['peluang lulus G3'], 0, None)], capsize=4)
    for x_, v, n in zip(xpos, d['peluang lulus G3'], d['n']):
        ax.text(x_, v + 0.06, f'{pct(v)}\nn={n}', ha='center', fontsize=9)
ax.set_xticks(np.arange(2))
ax.set_xticklabels(['Lulus di G2', 'Masih gagal di G2'])
ax.set_ylim(0, 1.15)
ax.set_ylabel('Peluang lulus G3')
ax.set_title('Checkpoint G2: peluang lulus G3 bagi siswa yang gagal di G1')
ax.legend()
simpan(fig, 'checkpoint_g2')

## Tahap 11. Model multivariabel: G1 + faktor kunci
**Tujuan:** memastikan faktor kunci tetap berkaitan dengan comeback ketika G1 dan faktor kunci lain dikendalikan bersamaan.

**Aturan jumlah variabel:** sekitar 10 kejadian per variabel (events per variable). Jumlah faktor yang boleh masuk = (jumlah kelompok terkecil / 10) dikurangi 1 slot untuk G1. Hanya faktor kunci non-nominal yang dipakai agar setiap faktor memakai satu koefisien.

In [ ]:
for nama, H in HASIL.items():
    R = H['ringkas']
    y = H['y'].to_numpy()
    kejadian = int(min(y.sum(), len(y) - y.sum()))
    k_max = max(0, kejadian // 10 - 1)
    kandidat = R[(R.status == 'Lolos') & (R.tipe != 'nominal')].faktor.tolist()
    pilih = kandidat[:k_max]
    print(f'\n=== {nama}: kelompok terkecil = {kejadian} siswa, maksimal {k_max} faktor + G1 ===')
    H['multivariabel'] = None
    if not pilih:
        print('Tidak ada faktor kunci non-nominal yang memenuhi syarat, model multivariabel dilewati.')
        continue
    blok = [pd.DataFrame({'const': 1.0, 'G1': H['biner']['G1'].astype(float)}, index=H['X'].index)]
    satuan = {'G1': 'per 1 poin'}
    for f in pilih:
        D, s = kodekan_logit(H['X'], f)
        blok.append(D)
        satuan[f] = s
    Z = pd.concat(blok, axis=1)
    try:
        m = sm.Logit(y, Z).fit(disp=0, maxiter=200)
        ci = m.conf_int()
        T = pd.DataFrame({'variabel': Z.columns[1:], 'satuan': [satuan.get(c, '') for c in Z.columns[1:]],
                          'OR': np.exp(m.params.iloc[1:]).values, 'CI bawah': np.exp(ci.iloc[1:, 0]).values,
                          'CI atas': np.exp(ci.iloc[1:, 1]).values, 'p': m.pvalues.iloc[1:].values})
        H['multivariabel'], H['pseudo_r2'] = T, m.prsquared
        display(T.round(3))
        print(f'McFadden pseudo R2 = {m.prsquared:.3f}. OR > 1 berarti peluang comeback lebih besar.')
        simpan_tabel(T, f'logistik_multivariabel_{KODE[nama]}')
    except Exception as e:
        print('Model gagal diestimasi:', type(e).__name__, e)

## Tahap 12. Model peringatan dini (klasifikasi dan prediksi)
**Tujuan:** menguji apakah faktor non-nilai menambah kemampuan memprediksi siswa yang akan **tetap gagal di G3**, dihitung saat G1 baru keluar.

**Desain anti leakage**
- Populasi: semua siswa G1 < 10. Target: 1 = gagal di G3 (perlu intervensi), 0 = lulus di G3.
- Fitur: G1 dan faktor non-nilai saja. **G2 dan G3 tidak dipakai.**
- Standardisasi dan one-hot encoding berada di dalam pipeline, sehingga dipelajari ulang di setiap fold.
- Evaluasi: repeated stratified k-fold (5 fold x 10 ulangan). Semua model memakai fold yang sama sehingga bisa dibandingkan berpasangan.

**Model:** baseline mayoritas, logistik G1 saja (baseline utama), logistik L2 (G1 + semua faktor), decision tree dangkal (aturan yang bisa dibaca guru), dan random forest sebagai pembanding model kompleks.

**Metrik:** ROC-AUC (tidak bergantung ambang), balanced accuracy (adil untuk kelas tidak seimbang), recall dan presisi kelompok gagal.

**Tingkat kepentingan faktor** dihitung dengan permutation importance pada data uji di setiap fold (penurunan AUC saat faktor diacak). SHAP tidak dipakai karena sampel kecil membuat atribusinya tidak stabil, dan pada model logistik hasilnya hanya mengulang koefisien.

In [ ]:
NAMA_MODEL = ['Baseline mayoritas', 'Logistik: G1 saja', 'Logistik L2: G1 + faktor',
              'Decision tree (kedalaman 3)', 'Random forest (pembanding)']
METRIK = ['AUC', 'Balanced accuracy', 'Recall gagal', 'Presisi gagal']


def buat_model(num, kat):
    def prep(skala=True):
        trans = [('num', StandardScaler() if skala else 'passthrough', num)]
        if kat:
            trans.append(('kat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), kat))
        return ColumnTransformer(trans)
    seed = CONFIG['SEED']
    return {
        'Baseline mayoritas': Pipeline([('prep', prep()), ('clf', DummyClassifier(strategy='prior'))]),
        'Logistik: G1 saja': Pipeline([('prep', ColumnTransformer([('num', StandardScaler(), ['G1'])])),
                                       ('clf', LogisticRegression(class_weight='balanced', max_iter=5000))]),
        'Logistik L2: G1 + faktor': Pipeline([('prep', prep()), ('clf', LogisticRegression(
            C=CONFIG['C_LOGISTIK'], class_weight='balanced', max_iter=5000))]),
        'Decision tree (kedalaman 3)': Pipeline([('prep', prep(False)), ('clf', DecisionTreeClassifier(
            max_depth=3, min_samples_leaf=8, class_weight='balanced', random_state=seed))]),
        'Random forest (pembanding)': Pipeline([('prep', prep(False)), ('clf', RandomForestClassifier(
            n_estimators=200, min_samples_leaf=5, class_weight='balanced', random_state=seed, n_jobs=-1))]),
    }


def evaluasi_peringatan_dini(H):
    fokus, Xf, aktif = H['fokus'], H['X_fokus'], H['aktif']
    Xp = pd.concat([fokus[['G1']].astype(float), Xf[aktif]], axis=1)
    y = (fokus['G3'] < CONFIG['AMBANG_LULUS']).astype(int).to_numpy()
    num = ['G1'] + [f for f in aktif if TIPE[f] != 'nominal']
    kat = [f for f in aktif if TIPE[f] == 'nominal']
    model = buat_model(num, kat)
    cv = RepeatedStratifiedKFold(n_splits=CONFIG['CV_FOLD'], n_repeats=CONFIG['CV_ULANG'],
                                 random_state=CONFIG['SEED'])
    skor = {m: {k: [] for k in METRIK} for m in model}
    imp = []
    for tr, te in cv.split(Xp, y):
        Xtr, Xte, ytr, yte = Xp.iloc[tr], Xp.iloc[te], y[tr], y[te]
        for m, est in model.items():
            e = clone(est).fit(Xtr, ytr)
            prob = e.predict_proba(Xte)[:, 1]
            pred = e.predict(Xte)
            skor[m]['AUC'].append(roc_auc_score(yte, prob))
            skor[m]['Balanced accuracy'].append(balanced_accuracy_score(yte, pred))
            skor[m]['Recall gagal'].append(recall_score(yte, pred, zero_division=0))
            skor[m]['Presisi gagal'].append(precision_score(yte, pred, zero_division=0))
            if m == 'Logistik L2: G1 + faktor':
                pi = permutation_importance(e, Xte, yte, scoring='roc_auc', n_repeats=5,
                                            random_state=CONFIG['SEED'])
                imp.append(pi.importances_mean)
    ringkas = pd.DataFrame({m: {k: np.mean(v) for k, v in s.items()} for m, s in skor.items()}).T
    sd = pd.DataFrame({m: {k: np.std(v) for k, v in s.items()} for m, s in skor.items()}).T
    auc_base = np.array(skor['Logistik: G1 saja']['AUC'])
    banding = {m: dict(delta_auc=np.mean(np.array(skor[m]['AUC']) - auc_base),
                       persen_fold_lebih_baik=np.mean(np.array(skor[m]['AUC']) > auc_base))
               for m in model if m != 'Logistik: G1 saja'}
    imp = np.array(imp)
    importance = pd.DataFrame({'fitur': Xp.columns, 'importance_mean': imp.mean(0), 'importance_sd': imp.std(0)})
    importance = importance.sort_values('importance_mean', ascending=False).reset_index(drop=True)
    pohon = clone(model['Decision tree (kedalaman 3)']).fit(Xp, y)
    return dict(skor=skor, ringkas=ringkas, sd=sd, banding=pd.DataFrame(banding).T, importance=importance,
                pohon=pohon, Xp=Xp, y_risiko=y)


def aturan_pohon(pipe, Xp, y):
    # Mengubah decision tree menjadi aturan "JIKA ... MAKA" dengan batas yang sudah diringkas
    tree, prep = pipe.named_steps['clf'], pipe.named_steps['prep']
    fitur = [n.split('__', 1)[-1] for n in prep.get_feature_names_out()]
    daun = tree.apply(prep.transform(Xp))
    t = tree.tree_
    kat_cols = [f for f in Xp.columns if TIPE.get(f) == 'nominal']

    def susun(syarat):
        batas, teks = {}, []
        for f, kiri, thr in syarat:
            if TIPE.get(f) == 'biner':
                teks.append(f'{f} = {LEVEL_NEGATIF[f] if kiri else LEVEL_POSITIF[f]}')
                continue
            induk = next((k for k in kat_cols if f.startswith(k + '_')), None)
            if induk:
                lv = f[len(induk) + 1:]
                teks.append(f'{induk} bukan {lv}' if kiri else f'{induk} = {lv}')
                continue
            lo, hi = batas.get(f, (None, None))
            b = int(np.floor(thr))
            if kiri:
                hi = b if hi is None else min(hi, b)
            else:
                lo = b + 1 if lo is None else max(lo, b + 1)
            batas[f] = (lo, hi)
        num = []
        for f, (lo, hi) in batas.items():
            if lo is not None and hi is not None:
                num.append(f'{f} = {lo}' if lo == hi else f'{lo} <= {f} <= {hi}')
            elif lo is not None:
                num.append(f'{f} >= {lo}')
            else:
                num.append(f'{f} <= {hi}')
        return ' DAN '.join(num + teks) if (num or teks) else 'semua siswa'

    hasil = []

    def telusur(node, syarat):
        if t.children_left[node] == t.children_right[node]:
            m = daun == node
            hasil.append(dict(aturan=susun(syarat), n_siswa=int(m.sum()),
                              proporsi_gagal_G3=float(y[m].mean()) if m.any() else np.nan))
            return
        f, thr = fitur[t.feature[node]], t.threshold[node]
        telusur(t.children_left[node], syarat + [(f, True, thr)])
        telusur(t.children_right[node], syarat + [(f, False, thr)])

    telusur(0, [])
    return pd.DataFrame(hasil).sort_values('proporsi_gagal_G3', ascending=False).reset_index(drop=True)


for nama, H in HASIL.items():
    print(f'Melatih dan mengevaluasi model untuk {nama} ...')
    P = evaluasi_peringatan_dini(H)
    P['aturan'] = aturan_pohon(P['pohon'], P['Xp'], P['y_risiko'])
    H['prediksi'] = P
    tabel = P['ringkas'].round(3).astype(str) + ' ± ' + P['sd'].round(3).astype(str)
    print(f'\n=== {nama}: rerata ± sd dari {CONFIG["CV_FOLD"] * CONFIG["CV_ULANG"]} fold '
          f'(target 1 = gagal di G3, n = {len(P["y_risiko"])}, gagal = {int(P["y_risiko"].sum())}) ===')
    display(tabel.loc[NAMA_MODEL])
    print('Perbandingan berpasangan terhadap "Logistik: G1 saja":')
    display(P['banding'].round(3))
    print('Aturan peringatan dini dari decision tree yang dilatih pada seluruh data (deskriptif). '
          'Proporsi gagal G3 adalah data aktual; kemampuan prediksi sebenarnya lihat AUC validasi silang.')
    display(P['aturan'].round(3))
    simpan_tabel(P['ringkas'].reset_index().rename(columns={'index': 'model'}), f'prediksi_ringkas_{KODE[nama]}')
    simpan_tabel(P['importance'], f'permutation_importance_{KODE[nama]}')
    simpan_tabel(P['aturan'], f'aturan_pohon_{KODE[nama]}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8), sharex=True)
for ax, (nama, H) in zip(axes, HASIL.items()):
    d = pd.DataFrame([(m, v) for m in NAMA_MODEL for v in H['prediksi']['skor'][m]['AUC']], columns=['model', 'AUC'])
    sns.boxplot(data=d, y='model', x='AUC', order=NAMA_MODEL, color='#90CAF9', showfliers=False, ax=ax)
    ax.axvline(0.5, ls='--', c='grey', lw=1)
    ax.set_title(f'ROC-AUC per fold (validasi silang): {nama}')
    ax.set_ylabel('')
fig.tight_layout()
simpan(fig, 'prediksi_auc_model')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (nama, H) in zip(axes, HASIL.items()):
    I = H['prediksi']['importance'].head(12).iloc[::-1]
    warna = ['#616161' if f == 'G1' else ('#2E7D32' if v > 0 else '#BDBDBD') for f, v in zip(I.fitur, I.importance_mean)]
    ax.barh(I.fitur, I.importance_mean, xerr=I.importance_sd, color=warna, capsize=3)
    ax.axvline(0, c='k', lw=1)
    ax.set_title(f'Permutation importance (penurunan AUC): {nama}')
    ax.set_xlabel('Rerata penurunan AUC saat fitur diacak (abu-abu tua = G1)')
fig.tight_layout()
simpan(fig, 'prediksi_permutation_importance')

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
for ax, (nama, H) in zip(axes, HASIL.items()):
    Rr = H['prediksi']['aturan'].iloc[::-1]
    label = [a if len(a) <= 70 else a[:67] + '...' for a in Rr.aturan]
    ax.barh(label, Rr.proporsi_gagal_G3, color=[plt.cm.Reds(0.3 + 0.6 * v) for v in Rr.proporsi_gagal_G3])
    for i, (v, n) in enumerate(zip(Rr.proporsi_gagal_G3, Rr.n_siswa)):
        ax.text(v + 0.01, i, f'{pct(v)} (n={n})', va='center', fontsize=8)
    ax.set_xlim(0, 1.25)
    ax.set_title(f'Aturan peringatan dini (deskriptif): proporsi gagal G3 per aturan, {nama}\n'
                 'kemampuan prediksi sebenarnya: lihat AUC validasi silang', fontsize=10)
    ax.tick_params(axis='y', labelsize=8)
fig.tight_layout()
simpan(fig, 'prediksi_aturan_pohon')

## Tahap 13. Perbandingan Matematika dan Portugis
**Tujuan:** menguji apakah faktor comeback berlaku umum atau khas mata pelajaran. Perbandingan memakai **ukuran efek**, bukan p-value, karena jumlah siswa di kedua dataset berbeda. Siswa yang sama di kedua file tidak digabung.

| Kategori | Arti |
|---|---|
| Konsisten | Lolos di kedua dataset dengan arah sama |
| Khas Matematika / Khas Portugis | Lolos hanya di satu dataset |
| Berlawanan | Lolos di keduanya tetapi arahnya berbeda |
| Tidak bermakna | Tidak lolos di keduanya |

In [ ]:
Tm = HASIL['Matematika']['ringkas'].set_index('faktor')
Tp = HASIL['Portugis']['ringkas'].set_index('faktor')
semua = sorted(set(Tm.index) | set(Tp.index), key=lambda f: FAKTOR.index(f))
rows = []
for f in semua:
    em, ep = Tm.efek.get(f, np.nan), Tp.efek.get(f, np.nan)
    sm_, sp_ = Tm.status.get(f, 'tidak diuji'), Tp.status.get(f, 'tidak diuji')
    lm, lp = sm_ == 'Lolos', sp_ == 'Lolos'
    searah = (TIPE[f] == 'nominal') or (np.sign(em) == np.sign(ep))
    if lm and lp:
        kategori = 'Konsisten' if searah else 'Berlawanan'
    elif lm:
        kategori = 'Khas Matematika'
    elif lp:
        kategori = 'Khas Portugis'
    else:
        kategori = 'Tidak bermakna'
    rows.append(dict(faktor=f, tipe=TIPE[f], peran=PERAN[f], efek_mat=em, status_mat=sm_,
                     skor_mat=Tm.skor_konsistensi.get(f, np.nan), efek_por=ep, status_por=sp_,
                     skor_por=Tp.skor_konsistensi.get(f, np.nan), kategori=kategori))
BANDING = pd.DataFrame(rows)
urutan_kat = ['Konsisten', 'Berlawanan', 'Khas Matematika', 'Khas Portugis', 'Tidak bermakna']
BANDING['_u'] = BANDING.kategori.map({k: i for i, k in enumerate(urutan_kat)})
BANDING = BANDING.sort_values(['_u', 'efek_mat'], key=lambda s: s if s.name == '_u' else -s.abs()).drop(columns='_u')
display(BANDING.round(3))
simpan_tabel(BANDING, 'perbandingan_mat_por')

D = BANDING.assign(urut=BANDING.efek_mat.fillna(0)).sort_values('urut').reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, 11))
ypos = np.arange(len(D))
ax.hlines(ypos, D[['efek_mat', 'efek_por']].min(1), D[['efek_mat', 'efek_por']].max(1), color='#BDBDBD', lw=2)
for kol, st, nama in [('efek_mat', 'status_mat', 'Matematika'), ('efek_por', 'status_por', 'Portugis')]:
    terisi = D[st] == 'Lolos'
    ax.scatter(D.loc[terisi, kol], ypos[terisi], color=WARNA[nama], s=60, zorder=3, label=f'{nama} (lolos)')
    ax.scatter(D.loc[~terisi, kol], ypos[~terisi], facecolors='white', edgecolors=WARNA[nama], s=60, zorder=3,
               label=f'{nama} (tidak lolos)')
ax.set_yticks(ypos)
ax.set_yticklabels([f'{f}  [{k}]' for f, k in zip(D.faktor, D.kategori)], fontsize=9)
for v in (-E, E):
    ax.axvline(v, ls='--', c='grey', lw=0.9)
ax.axvline(0, c='k', lw=1)
ax.set_xlabel('Ukuran efek (+ = lebih tinggi / lebih sering pada Comeback)')
ax.set_title('Perbandingan ukuran efek: Matematika vs Portugis')
ax.legend(loc='lower right', fontsize=8)
simpan(fig, 'perbandingan_efek_mat_por')

## Tahap 14. Peringkat faktor final
Tiga peringkat dilaporkan berdampingan karena menjawab pertanyaan berbeda:
- **Peringkat signifikansi** (q terkecil): seberapa meyakinkan perbedaan itu bukan kebetulan.
- **Peringkat kekuatan** (|efek| terbesar): seberapa besar perbedaan antar kelompok.
- **Peringkat kepentingan model** (permutation importance): seberapa besar kontribusi faktor dalam memprediksi kegagalan di luar G1.

Urutan akhir memakai **skor konsistensi**, lalu |efek|.

In [ ]:
for nama, H in HASIL.items():
    R = H['ringkas'].copy()
    imp = H['prediksi']['importance'].set_index('fitur')['importance_mean']
    R['importance'] = R.faktor.map(imp)
    R['rank_signifikansi'] = 0
    R.loc[R.sort_values(['q', 'p']).index, 'rank_signifikansi'] = np.arange(1, len(R) + 1)
    R['rank_kekuatan'] = R.efek_abs.rank(ascending=False, method='min').astype(int)
    R['rank_kepentingan'] = R.importance.fillna(-np.inf).rank(ascending=False, method='min').astype(int)
    R['kategori_banding'] = R.faktor.map(BANDING.set_index('faktor').kategori)
    R = R.sort_values(['skor_konsistensi', 'efek_abs'], ascending=False).reset_index(drop=True)
    R.insert(0, 'peringkat', np.arange(1, len(R) + 1))
    H['peringkat'] = R
    kolom = ['peringkat', 'faktor', 'peran', 'efek', 'q', 'status', 'skor_konsistensi', 'importance',
             'rank_signifikansi', 'rank_kekuatan', 'rank_kepentingan', 'kategori_banding']
    print(f'\n=== Peringkat faktor final: {nama} ===')
    display(R[kolom].round(4))
    simpan_tabel(R, f'peringkat_faktor_{KODE[nama]}')

## Tahap 15. Kesimpulan otomatis
Kesimpulan disusun langsung dari hasil di atas: pola transisi, faktor terpenting (signifikansi, kekuatan, kepentingan model, konsistensi), perbandingan kedua dataset, evaluasi model, temuan utama, keterbatasan, dan rekomendasi. Teks juga disimpan sebagai `kesimpulan_otomatis.txt`.

In [ ]:
def deskripsi_faktor(r):
    f = r['faktor']
    nm = NAMA.get(f, f)
    if r['tipe'] in ('ordinal', 'numerik'):
        arah = 'lebih rendah' if r['efek'] < 0 else 'lebih tinggi'
        return f'{nm} ({f}) {arah} pada siswa Comeback (r = {angka(r["efek"])}, {teks_p("q", r["q"])})'
    if r['tipe'] == 'biner':
        arah = 'lebih sering' if r['efek'] > 0 else 'lebih jarang'
        return (f'siswa dengan {nm} ({f} = {LEVEL_POSITIF[f]}) {arah} menjadi Comeback '
                f'(phi = {angka(r["efek"])}, {teks_p("q", r["q"])})')
    return f'{nm} ({f}) berbeda distribusinya antar kelompok (V = {angka(r["efek"])}, {teks_p("q", r["q"])})'


KATA_ARAH = {'age': ('lebih tua', 'lebih muda'), 'failures': ('lebih banyak', 'lebih sedikit'),
             'absences': ('tinggi', 'rendah'), 'traveltime': ('lebih lama', 'lebih singkat')}


def rekomendasi_faktor(r):
    f = r['faktor']
    nm = NAMA.get(f, f)
    if r['tipe'] in ('ordinal', 'numerik'):
        tinggi, rendah = KATA_ARAH.get(f, ('tinggi', 'rendah'))
        teks = f'Siswa gagal G1 dengan {nm} {tinggi if r["efek"] < 0 else rendah} lebih jarang bangkit'
    elif r['tipe'] == 'biner':
        lv = LEVEL_NEGATIF[f] if r['efek'] > 0 else LEVEL_POSITIF[f]
        teks = f'Siswa gagal G1 dengan {f} = {lv} lebih jarang bangkit'
    else:
        teks = f'Peluang bangkit berbeda menurut {nm}'
    if r['peran'] == 'dapat diintervensi':
        return (teks + f'. Jadikan {nm} indikator pemantauan sejak G1 keluar, dan uji intervensinya '
                       'sebagai hipotesis, bukan kepastian.')
    return teks + f'. Faktor {nm} tidak bisa diubah sekolah, jadi gunakan hanya untuk mengenali siswa berisiko.'


def susun_kesimpulan():
    L = []
    tulis = L.append
    tulis('KESIMPULAN OTOMATIS: ANALISIS TRANSISI NILAI G1 → G2 → G3')
    tulis('Seluruh temuan bersifat asosiasi dari data observasional, bukan hubungan sebab-akibat.')

    tulis('\n1. Data dan pembersihan')
    for _, r in LOG.iterrows():
        tulis(f'- {r.dataset}: {r.n_awal} siswa awal, {r.total_dibuang} dikeluarkan ({r.nilai_0_dibuang} karena '
              f'nilai 0 di salah satu periode), {r.n_akhir} dianalisis. Siswa yang gagal di G1: '
              f'{len(HASIL[r.dataset]["fokus"])}.')

    tulis('\n2. Pola transisi nilai')
    for nama, H in HASIL.items():
        f_ = H['fokus']
        n = len(f_)
        vc = f_.kategori.value_counts()
        awal, akhir = int(vc.get('Comeback Awal', 0)), int(vc.get('Comeback Akhir', 0))
        sem, gagal = int(vc.get('Comeback Sementara', 0)), int(vc.get('Tetap Gagal', 0))
        cb = awal + akhir
        porsi_awal = awal / cb if cb else np.nan
        if porsi_awal >= 0.6:
            waktu = f'Sebagian besar comeback terjadi lebih awal, sudah lulus di G2 ({pct(porsi_awal)} dari comeback).'
        elif porsi_awal <= 0.4:
            waktu = f'Sebagian besar comeback baru terjadi di periode akhir ({pct(1 - porsi_awal)} dari comeback).'
        else:
            waktu = (f'Waktu comeback terbagi hampir seimbang: {pct(porsi_awal)} sudah lulus di G2 dan '
                     f'{pct(1 - porsi_awal)} baru lulus di G3.')
        d = H['df']
        tergelincir = (d.loc[d.lulus_G1, 'kategori'] == 'Tergelincir').mean()
        tulis(f'- {nama}: dari {n} siswa yang gagal di G1, {cb} ({pct(cb / n)}) lulus di G3, terdiri dari '
              f'Comeback Awal {awal} dan Comeback Akhir {akhir}. Comeback Sementara {sem} ({pct(sem / n)}) dan '
              f'Tetap Gagal {gagal} ({pct(gagal / n)}). {waktu} '
              f'Sebaliknya, {pct(tergelincir)} siswa yang lulus di G1 tergelincir gagal di G3.')

    tulis('\n3. Titik awal G1')
    for nama, H in HASIL.items():
        g = H['g1_awal']
        catatan = ('Titik awal berbeda, sehingga hasil faktor wajib dibaca bersama kontrol G1 (strata dan regresi '
                   'logistik).' if g['p'] < A else 'Titik awal tidak berbeda bermakna, tetapi kontrol G1 tetap dilakukan.')
        tulis(f'- {nama}: median G1 Comeback {angka(g["med1"], 0)} vs Tetap Gagal {angka(g["med0"], 0)} '
              f'({teks_p("p", g["p"])}, r = {angka(g["r"])}). {catatan}')

    tulis('\n4. G2 sebagai checkpoint')
    for nama, H in HASIL.items():
        g = H['g2']
        tulis(f'- {nama}: siswa gagal G1 yang sudah lulus di G2 berpeluang lulus G3 sebesar '
              f'{pct(g.loc["Lulus G2", "peluang lulus G3"])} (n = {g.loc["Lulus G2", "n"]}), dibanding '
              f'{pct(g.loc["Gagal G2", "peluang lulus G3"])} (n = {g.loc["Gagal G2", "n"]}) bagi yang masih gagal '
              f'di G2 (uji Fisher {teks_p("p", H["g2_p"])}).')

    tulis(f'\n5. Faktor terpenting (kriteria: q < {angka(A)} dan |efek| >= {angka(E)}, '
          'diurutkan menurut skor konsistensi 0 sampai 6)')
    for nama, H in HASIL.items():
        R = H['peringkat']
        kunci = R[R.status == 'Lolos']
        if len(kunci):
            tulis(f'- {nama}: {len(kunci)} faktor kunci.')
            for i, (_, r) in enumerate(kunci.iterrows(), 1):
                tulis(f'   {i}) {deskripsi_faktor(r)}; skor konsistensi {r.skor_konsistensi}/6; {r.peran}.')
        else:
            top = R.nlargest(3, 'efek_abs')
            tulis(f'- {nama}: tidak ada faktor yang lolos kriteria. Efek terbesar (eksploratif): '
                  + '; '.join(deskripsi_faktor(r) for _, r in top.iterrows()) + '.')
        ind = R[R.status == 'Indikasi'].faktor.tolist()
        if ind:
            tulis(f'   Indikasi (q < {angka(AI)}): {daftar(ind)}.')
        sig = R.sort_values(['q', 'p']).head(5)
        tulis('   Peringkat signifikansi (q terkecil): '
              + ', '.join(f'{r.faktor} (q {fmt_q(r.q)})' for _, r in sig.iterrows()) + '.')
        kuat = R.sort_values('efek_abs', ascending=False).head(5)
        tulis('   Peringkat kekuatan hubungan (|efek| terbesar): '
              + ', '.join(f'{r.faktor} ({angka(r.efek)})' for _, r in kuat.iterrows()) + '.')
        imp = R[R.importance > 0].sort_values('importance', ascending=False).head(5)
        tulis('   Peringkat kepentingan model (penurunan AUC saat diacak): '
              + (', '.join(f'{r.faktor} ({angka(r.importance, 3)})' for _, r in imp.iterrows()) if len(imp)
                 else 'tidak ada faktor yang menurunkan AUC saat diacak') + '.')
        st = R.set_index('faktor').status
        umum = [f for f in ['studytime', 'paid', 'schoolsup', 'famsup', 'internet', 'higher']
                if f in st.index and st[f] == 'Tidak bermakna']
        if umum:
            tulis(f'   Faktor yang sering dianggap penting tetapi tidak membedakan kedua kelompok: {daftar(umum)}.')
        if H.get('pasangan_kuat'):
            tulis('   Pasangan faktor berkorelasi kuat (dibaca sebagai satu temuan): '
                  + ', '.join(f'{a} & {b} (rho = {angka(v)})' for a, b, v in H['pasangan_kuat']) + '.')

    tulis('\n6. Perbandingan Matematika dan Portugis')
    for k in ['Konsisten', 'Berlawanan', 'Khas Matematika', 'Khas Portugis']:
        tulis(f'- {k}: {daftar(BANDING[BANDING.kategori == k].faktor.tolist())}.')
    laju = {n: H['fokus'].kategori.isin(['Comeback Awal', 'Comeback Akhir']).mean() for n, H in HASIL.items()}
    tulis(f'- Tingkat comeback: Matematika {pct(laju["Matematika"])}, Portugis {pct(laju["Portugis"])}. '
          'Perbandingan ini deskriptif karena sebagian siswa tercatat di kedua dataset.')

    tulis(f'\n7. Model peringatan dini (validasi silang {CONFIG["CV_FOLD"]}-fold x {CONFIG["CV_ULANG"]} ulangan, '
          'target = gagal di G3, fitur tanpa G2 dan G3)')
    for nama, H in HASIL.items():
        P = H['prediksi']
        Rk, Bd = P['ringkas'], P['banding']
        kandidat = Rk.drop(index='Baseline mayoritas')
        terbaik = kandidat.AUC.idxmax()
        tulis(f'- {nama}: AUC model G1 saja {angka(Rk.loc["Logistik: G1 saja", "AUC"], 3)}. AUC tertinggi dicapai '
              f'{terbaik.split(" (")[0].lower()} sebesar {angka(kandidat.AUC.max(), 3)} dengan balanced accuracy '
              f'{angka(kandidat.loc[terbaik, "Balanced accuracy"], 3)}.')
        d = Bd.loc['Logistik L2: G1 + faktor']
        if d.delta_auc >= 0.03 and d.persen_fold_lebih_baik >= 0.7:
            nilai = 'faktor non-nilai memberi tambahan kemampuan prediksi yang konsisten di luar G1'
        elif d.delta_auc > 0:
            nilai = 'tambahan dari faktor non-nilai kecil dan belum konsisten'
        else:
            nilai = 'faktor non-nilai tidak menambah kemampuan prediksi di luar G1'
        tulis(f'  Logistik L2 vs G1 saja: selisih AUC {angka(d.delta_auc, 3)}, lebih baik di '
              f'{pct(d.persen_fold_lebih_baik)} fold, artinya {nilai}.')
        rf = Rk.loc['Random forest (pembanding)', 'AUC']
        log_terbaik = Rk.loc[['Logistik: G1 saja', 'Logistik L2: G1 + faktor'], 'AUC']
        if rf <= log_terbaik.max() + 0.01:
            tulis(f'  Random forest (AUC {angka(rf, 3)}) tidak lebih baik dari model logistik terbaik '
                  f'(AUC {angka(log_terbaik.max(), 3)}), sehingga model kompleks tidak diperlukan untuk data sebesar ini.')
        else:
            tulis(f'  Random forest sedikit lebih tinggi dari model logistik terbaik (AUC {angka(rf, 3)} vs '
                  f'{angka(log_terbaik.max(), 3)}), tetapi lebih sulit ditafsirkan dan rawan tidak stabil pada sampel kecil.')
        at = P['aturan'].iloc[0]
        tulis(f'  Aturan risiko tertinggi dari decision tree: {at.aturan} (n = {at.n_siswa}, '
              f'{pct(at.proporsi_gagal_G3)} gagal di G3).')

    tulis('\n8. Temuan utama')
    peta = BANDING.set_index('faktor').kategori
    for nama, H in HASIL.items():
        f_ = H['fokus']
        n = len(f_)
        cb = int(f_.kategori.isin(['Comeback Awal', 'Comeback Akhir']).sum())
        kunci = H['peringkat'][H['peringkat'].status == 'Lolos']
        if len(kunci):
            r = kunci.iloc[0]
            kat = peta.get(r.faktor, '-')
            searah_lain = bool(H['konsistensi'].loc[r.faktor, LAPIS[5]])
            if kat == 'Konsisten':
                lanjut = f'dan pola yang sama juga lolos kriteria di {LAIN[nama]}'
            elif kat == 'Berlawanan':
                lanjut = f'tetapi arahnya berlawanan di {LAIN[nama]}'
            elif searah_lain:
                lanjut = f'dengan arah yang sama di {LAIN[nama]} tetapi lebih lemah dan tidak lolos kriteria'
            else:
                lanjut = f'sedangkan di {LAIN[nama]} pola ini tidak muncul'
            tulis(f'- {nama}: dari {n} siswa yang gagal di G1, {cb} ({pct(cb / n)}) berhasil lulus. Faktor yang paling '
                  f'konsisten membedakan mereka: {deskripsi_faktor(r)}, {lanjut}.')
        else:
            tulis(f'- {nama}: dari {n} siswa yang gagal di G1, {cb} ({pct(cb / n)}) berhasil lulus. Tidak ada faktor '
                  'non-nilai yang lolos kriteria, sehingga lintasan nilai (G1 dan G2) menjadi penanda terkuat.')

    tulis('\n9. Keterbatasan')
    tulis('- Data survei satu waktu dari dua sekolah di Portugal (tahun ajaran 2005/2006), sehingga temuan bersifat '
          'asosiasi dan generalisasinya terbatas.')
    tulis('- Ukuran kelompok kecil: ' + '; '.join(
        f'{n} {int(H["y"].sum())} Comeback vs {int((H["y"] == 0).sum())} Tetap Gagal' for n, H in HASIL.items())
          + '. Kekuatan uji terbatas dan interval kepercayaan lebar.')
    tulis(f'- Sekitar {len(HASIL["Matematika"]["aktif"])} faktor diuji sekaligus. Koreksi Benjamini-Hochberg '
          'mengurangi temuan palsu, tetapi tidak menghilangkannya.')
    tulis('- Faktor seperti goout, Dalc, Walc, studytime, dan freetime dilaporkan sendiri oleh siswa, sehingga rentan bias.')
    tulis('- Siswa dengan nilai 0 dikeluarkan, sehingga temuan tidak berlaku untuk siswa yang berhenti atau tidak '
          'mengikuti ujian.')
    sem = {n: int((H['fokus'].kategori == 'Comeback Sementara').sum()) for n, H in HASIL.items()}
    tulis(f'- Comeback Sementara berjumlah kecil (Matematika {sem["Matematika"]}, Portugis {sem["Portugis"]}), '
          f'sehingga hanya ikut uji antar kategori jika jumlahnya minimal {CONFIG["MIN_N_KELOMPOK"]}.')

    tulis('\n10. Rekomendasi (hipotesis tindakan, bukan klaim sebab-akibat)')
    ada = False
    for nama, H in HASIL.items():
        R = H['peringkat']
        for _, r in R[(R.status == 'Lolos') & (R.skor_konsistensi >= 4)].head(3).iterrows():
            tulis(f'- [{nama}] {rekomendasi_faktor(r)}')
            ada = True
    if not ada:
        tulis('- Belum ada faktor non-nilai yang cukup konsisten (skor >= 4) untuk dijadikan dasar intervensi. '
              'Fokuskan pemantauan pada G1 dan G2.')
    for nama, H in HASIL.items():
        g = H['g2']
        p1, p0 = g.loc['Lulus G2', 'peluang lulus G3'], g.loc['Gagal G2', 'peluang lulus G3']
        if p1 - p0 >= 0.2:
            tulis(f'- [{nama}] Gunakan G2 sebagai checkpoint kedua: siswa gagal G1 yang masih gagal di G2 hanya '
                  f'berpeluang lulus {pct(p0)} (vs {pct(p1)}), sehingga perlu pendampingan sebelum ujian akhir.')
    tulis('- Gabungkan aturan decision tree (saat G1 keluar) dan status G2 menjadi sistem peringatan dini dua tahap, '
          'lalu uji efektivitas intervensinya dengan kelompok pembanding pada penelitian lanjutan.')

    teks = '\n'.join(L)
    with open(os.path.join(OUT, 'kesimpulan_otomatis.txt'), 'w', encoding='utf-8') as fh:
        fh.write(teks)
    return teks


print(susun_kesimpulan())

## Ekspor hasil
Semua grafik, tabel, dan kesimpulan dikemas menjadi `hasil_analisis.zip`. Di Colab, file otomatis terunduh.

In [ ]:
arsip = shutil.make_archive('hasil_analisis', 'zip', OUT)
print(f'{len(os.listdir(OUT))} file dikemas ke {arsip}')
try:
    from google.colab import files
    files.download(arsip)
except ImportError:
    pass